In [1]:
# ════════════════════════════════════════════════════════════════════════════════════════════
# ScoutIA — RUN « COMME EN PRODUCTION » SUR UN MATCH (11/11/2026)
# Vidéo brute → TOUT, dans une seule cellule, avec les réglages de production et AUCUNE réponse Gemini réutilisée :
#   ÉTAPE 1 — analyse du match : suivi des joueurs (GPU), coups d'envoi et bornes, détection des buts (validation KO en
#             résolution moyenne, temps forts en basse, escalade), tenues + attribution des buts aux équipes (normale,
#             1 image/s), résumé, rapport PDF client.
#   ÉTAPE 2 — montage d'un joueur (kaggle_joueur_v3_2.py, LECTURE RAPIDE, 12 images par morceau) : numéro lu par
#             PARSeq v2, relecture Gemini des cas presque trouvés, actions, tous les buts du match, vidéo montée.
#   BILAN — temps de chaque étape, coût Gemini réel de chaque étape, temps GPU, livrables.
# Réglages : MATCH (étape 1) ; NUMERO / COULEUR du joueur (étape 2, plus bas).
# Entrées : la vidéo, codeprojectai (avec parseq_scoutia_maillots_v2.pt), mziparameter. GPU T4 ×2, Internet activé.
# GitHub : pipeline.py, gemini_validator.py, ai/attribution_equipes.py, export/pdf.py à jour (contrôle avant le run).
# Lancer en « Save & Run All (Commit) ». Estimation (non mesurée) : ≈ 1 h 45 analyse + ≈ 20-30 min joueur ; ≈ 2-2,5 $.
# ════════════════════════════════════════════════════════════════════════════════════════════
import os, sys, shutil, logging, builtins, time

os.environ["DEBUG"] = "false"
os.environ["DISABLE_OCR"] = "1"
os.makedirs("/kaggle/working/outputs", exist_ok=True)

MATCH = "minerois_ster"
T0_BOUT_EN_BOUT = time.time()
import datetime as _dt_prod
_T_DEBUT_GEMINI = time.time()
SEGMENTS = True
SUIVI_SEULEMENT = False
REPLAY = False            # TEST GLOBAL : run complet, nouveau cache de suivi et cache Gemini à jour
KO_DEDUP = "fenetre"
CACHE_REPLAY = ""
SECOND_REGARD = True
OCCASIONS_KO = True
VOIE_BALLON = False
REFLEXION_REDUITE = True
LECTURE_MAILLOTS = False
KICKOFF_AUTO = True       # comme pour un client : coup d'envoi détecté par le pipeline

MATCHS = {'minerois_ster': {'video': '/kaggle/input/datasets/michaelziant/mineroisster-complet/minerois-ster-2025-11-09.mp4',
                            'camera_type': 'high_side',
                            'buts': ['55:42', '87:16', '99:02', '103:02', '110:08'],
                            'coup_envoi_s': 722.0, 'ko2_s': None},
          'raeren': {'video': '/kaggle/input/datasets/michaelziant/raeren-complet/ster-b-raeren-b-2026-04-03.mp4',
                     'camera_type': 'high_side',
                     'buts': ['16:39', '20:08', '27:18', '33:24', '45:25', '46:50', '50:17', '69:12', '91:00'],
                     'coup_envoi_s': 409.0, 'ko2_s': None}}
_M = MATCHS[MATCH]
LOG_FILE = f"/kaggle/working/outputs/{MATCH}_reference.log"
OUT_DIR = f"outputs/{MATCH}_reference"
GEMINI_CACHE = "on"
FILTRAGE_PISTES = ""         # suivi de PRODUCTION (historique)
KO_CONSIGNE, KO_FENETRE, KO_TEMPERATURE = "fenetre", "30", "0"

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s",
                    handlers=[logging.FileHandler(LOG_FILE, mode="w", encoding="utf-8")])
for noisy in ["PIL", "pytesseract", "ultralytics", "numba", "librosa", "absl", "torchreid", "matplotlib"]:
    logging.getLogger(noisy).setLevel(logging.WARNING)
for _h in logging.getLogger().handlers:
    _h.addFilter(lambda _r: not _r.name.startswith("numba"))
log = logging.getLogger()
if hasattr(builtins.print, "__wrapped__"):
    builtins.print = builtins.print.__wrapped__
_orig_print = builtins.print
def _print(*args, **kwargs):
    log.info(" ".join(str(a) for a in args))
    _orig_print(*args, **kwargs)
_print.__wrapped__ = _orig_print
builtins.print = _print

# ── Clés : lues dans le fichier param.env du dataset PRIVÉ « mziparameter » (plus aucune clé en clair ici) ──
#    à défaut : variables d'environnement, puis secrets Kaggle. Les clés ne sont jamais affichées.
PARAM_ENV = "/kaggle/input/datasets/michaelziant/mziparameter/param.env"
_PARAMS = {}
_pe = PARAM_ENV if os.path.exists(PARAM_ENV) else next(iter(__import__("glob").glob("/kaggle/input/**/mziparameter/**/param.env", recursive=True)), None)
if _pe:
    for _l in open(_pe, encoding="utf-8"):
        _l = _l.strip()
        if _l and not _l.startswith("#") and "=" in _l:
            _k, _v = _l.split("=", 1)
            _PARAMS[_k.strip()] = _v.strip().strip('"').strip("'")
    print(f"  Clés lues dans {_pe} : {', '.join(k for k in _PARAMS if k.endswith('_KEY'))}")
else:
    print(f"  ⚠️ {PARAM_ENV} introuvable — ajoute le dataset « mziparameter » en entrée")
def _secret(nom):
    v = _PARAMS.get(nom, "") or os.getenv(nom, "")
    if not v:
        try:
            from kaggle_secrets import UserSecretsClient
            v = UserSecretsClient().get_secret(nom) or ""
        except Exception:
            v = ""
    return v
GEMINI_KEY = _secret("GEMINI_API_KEY")
CLAUDE_KEY = _secret("CLAUDE_API_KEY")
if not GEMINI_KEY:
    raise SystemExit("❌ GEMINI_API_KEY introuvable (ni dans param.env de « mziparameter », ni dans les secrets) : run annulé")

print("\n" + "=" * 50 + "\nÉTAPE 1/3 — Clone\n" + "=" * 50)
os.chdir("/kaggle/working")
if os.path.exists("/kaggle/working/ProjectAIMika"):
    shutil.rmtree("/kaggle/working/ProjectAIMika")
if os.system("git clone --depth 1 https://github.com/mikarocco-droid/ProjectAIMika.git /kaggle/working/ProjectAIMika") != 0:
    raise SystemExit("❌ Clone échoué")
print("Clone OK")

print("\n" + "=" * 50 + "\nÉTAPE 2/3 — Dépendances\n" + "=" * 50)
os.system("pip install -q av torchreid flask flask-sqlalchemy flask-login ultralytics opencv-python-headless anthropic fpdf2 stripe "
          "python-dotenv werkzeug deep-sort-realtime scikit-learn google-genai joblib numpy psutil")
os.system("apt-get install -q tesseract-ocr ffmpeg")
os.system("pip install -q pytesseract")
print("Dépendances OK")

print("\n" + "=" * 50 + "\nÉTAPE 3/3 — Environnement\n" + "=" * 50)
os.chdir("/kaggle/working/ProjectAIMika")
sys.path.insert(0, "/kaggle/working/ProjectAIMika")
for d in ["uploads", "outputs", "instance", "models", ".env", "outputs/learning", "outputs/learning/football", OUT_DIR, OUT_DIR + "/cache"]:
    os.makedirs(d, exist_ok=True)
for pkg in ["analysis", "analytics", "ai", "vision", "video", "rendering", "export", "payments", "sports", "tracking"]:
    os.makedirs(pkg, exist_ok=True)
    if not os.path.exists(f"{pkg}/__init__.py"):
        open(f"{pkg}/__init__.py", "w").close()
with open(".env/param.env", "w") as f:
    f.write(f"""SECRET_KEY=test-kaggle
DATABASE_URI=sqlite:///db.sqlite3
DEBUG=false
UPLOAD_FOLDER=uploads
OUTPUT_FOLDER=outputs
FPS=25
FRAME_WIDTH=1920
FRAME_HEIGHT=1080
YOLO_MODEL=yolo26m.pt
YOLO_CONFIDENCE=0.4
BALL_METHOD=hybrid
CLAUDE_API_KEY={CLAUDE_KEY}
CLAUDE_MODEL=claude-sonnet-4-5
GEMINI_API_KEY={GEMINI_KEY}
R2_ENABLED=false
YOLO_BATCH_SIZE=4
YOLO_IMGSZ=640
FRAME_SKIP_EVERY=4
FILE_RETENTION_DAYS=30
""")
os.environ["GEMINI_API_KEY"] = GEMINI_KEY
os.environ["CLAUDE_API_KEY"] = CLAUDE_KEY
os.environ["DISABLE_OCR"] = "1"
os.environ["GEMINI_MAX_PARALLEL"] = "4"
os.environ.pop("TRACKER_MAX_PREDICTION_AGE", None)
os.environ["SCOUTIA_TRACKER_MODE"] = "historique"
os.environ["SCOUTIA_KO_CONSIGNE"] = KO_CONSIGNE
os.environ["SCOUTIA_KO_FENETRE"] = KO_FENETRE
os.environ["SCOUTIA_KO_TEMPERATURE"] = KO_TEMPERATURE
print("Environnement OK")


# ────────────────────────────────────────────────────────────

print("\n" + "=" * 50 + "\nCONTRÔLE PRÉ-VOL\n" + "=" * 50)
_R = "/kaggle/working/ProjectAIMika"
_src = open(f"{_R}/pipeline.py", encoding="utf-8").read()
_src_gv = open(f"{_R}/ai/gemini_validator.py", encoding="utf-8").read()
_src_main = open(f"{_R}/main.py", encoding="utf-8").read()
_src_pdf = open(f"{_R}/export/pdf.py", encoding="utf-8").read()
_checks = {
    "KO start_s explicite (replay OK)": "_ko_start_s = max(0.0, _kickoff_offset - 10.0)" in _src,
    "_ko_goals compare aux buts seulement": 'for g in _goals_deduped\n                if g.get("type") in ("goal", "score")' in _src,
    "pipeline : statut des buts + KO2 + temps forts": "_ko2_cand_idx" in _src and '"temps_forts":' in _src,
    "pipeline : contrôle final des doublons": "_temps_forts_final" in _src,
    "pipeline : mi-temps en parallèle (bornes)": "def _analyser_mi_temps(" in _src and "_DECALAGE_IDS_MT2" in _src,
    "pipeline : règle de la pause SÛRE (28/10)": "_MARGE_PAUSE_MI_TEMPS_S" in _src,
    "pipeline : voie « ballon » désactivée par défaut": '"SCOUTIA_VOIE_BALLON", "0"' in _src,
    "pipeline : second regard activé par défaut": '"SCOUTIA_KO_SECOND_REGARD", "1"' in _src,
    "pipeline : buts avec statut + identification": '"identification_joueurs":' in _src and "goals_a_verifier" in _src,
    "pipeline : doublons de temps forts quelle que soit l'issue (31/10)": "QUEL QUE SOIT le résultat vu par Gemini" in _src,
    "export/pdf : frise des Top Highlights sur plusieurs lignes (31/10)": "_fin_ligne" in _src_pdf,
    "gemini_validator : réflexion réduite (KO, classification)": "def _appliquer_reflexion_reduite(" in _src_gv,
    "gemini_validator : modèle unique + cache pour tous": "_installer_verrou_gemini" in _src_gv,
    "gemini_validator : cache par match": "SCOUTIA_GEMINI_CACHE_VIDEO" in _src_gv,
    "player_reid : version VALIDÉE": "linear_sum_assignment" not in open(f"{_R}/analysis/player_reid.py", encoding="utf-8").read(),
    "replay : le cache conserve time_since_update": '"time_since_update": p.get("time_since_update")' in open(f"{_R}/replay.py", encoding="utf-8").read(),
}
_ok = True
for _k, _v in _checks.items():
    print(f"  {'✅' if _v else '❌'} {_k}")
    _ok = _ok and _v
import glob as _g_prod
_att_mod = _g_prod.glob("/kaggle/working/ProjectAIMika/**/attribution_equipes.py", recursive=True)
_att_src = open(_att_mod[0], encoding="utf-8").read() if _att_mod else ""
for _nom, _v in {"ai/attribution_equipes.py présent (11/11 : normale + limite 30 Mo)": "requetes_recompressees" in _att_src,
                 "pipeline : attribution des équipes (Step 11b)": "Step 11b" in open(
                     _g_prod.glob("/kaggle/working/ProjectAIMika/**/pipeline.py", recursive=True)[0], encoding="utf-8").read()}.items():
    print(f"  {'✅' if _v else '❌'} {_nom}")
    _ok = _ok and _v
if not _ok:
    raise SystemExit("❌ La version sur GitHub n'est PAS la bonne — run annulé")
import torch as _torch
if not _torch.cuda.is_available():
    raise SystemExit("❌ GPU non activé : Settings → Accelerator → GPU")
print(f"  GPU : {_torch.cuda.get_device_name(0)}\n✅ Version correcte — lancement")


# ────────────────────────────────────────────────────────────

VIDEO = _M["video"]
if not os.path.exists(VIDEO):
    raise SystemExit(f"❌ Vidéo introuvable : {VIDEO}")
os.environ["SCOUTIA_GEMINI_CACHE"] = GEMINI_CACHE
os.environ["SCOUTIA_STOP_APRES_SUIVI"] = ""
os.environ["SCOUTIA_KO_DEDUP"] = KO_DEDUP
os.environ["SCOUTIA_KO_SECOND_REGARD"] = "1" if SECOND_REGARD else "0"
os.environ["SCOUTIA_TF_DESCRIPTIONS_KO"] = "1" if OCCASIONS_KO else ""
os.environ["SCOUTIA_VOIE_BALLON"] = "1" if VOIE_BALLON else "0"
os.environ["SCOUTIA_REFLEXION_REDUITE"] = "1" if REFLEXION_REDUITE else "0"
os.environ["SCOUTIA_LECTURE_MAILLOTS"] = "1" if LECTURE_MAILLOTS else "0"
os.environ["SCOUTIA_GEMINI_CACHE_OUT"] = f"/kaggle/working/gemini_cache_{MATCH}.jsonl"
os.environ["SCOUTIA_GEMINI_CACHE_IN"] = "/kaggle/working/aucun_cache_lu.jsonl"   # rien n'est relu → VRAI PRIX
os.environ["SCOUTIA_GEMINI_CACHE_VIDEO"] = VIDEO
import sys as _sys_purge
_racines = ("pipeline", "main", "config", "replay", "ai", "analysis", "vision", "rendering", "utils", "segment_extractor")
for _m in [m for m in list(_sys_purge.modules) if m.split(".")[0] in _racines]:
    del _sys_purge.modules[_m]
from pipeline import run_pipeline
print("\n" + "=" * 50 + f"\nANALYSE POUR LE JOUEUR — {MATCH} (suivi + bornes + buts)\n" + "=" * 50)
pipeline_t0 = time.time()
result = {}
try:
    result = run_pipeline(video_path=VIDEO, sport="football", output_dir=OUT_DIR, camera_type=_M["camera_type"],
                          use_single_yolo26=True, ball_model_name="yolo26m.pt", ball_conf=0.15, video_end_s=None,
                          kickoff_s_precalcule=None, use_match_boundary_segments=SEGMENTS, _match_data=None)
finally:
    print(f"\nPipeline runtime : {(time.time() - pipeline_t0) / 60:.1f} min")
    _cache_dir = f"/kaggle/working/ProjectAIMika/{OUT_DIR}/cache"
    if os.path.exists(os.path.join(_cache_dir, "meta.json")):
        shutil.make_archive(f"/kaggle/working/cache_{MATCH}_reference", "zip", _cache_dir)
        print(f"✅ Cache de suivi sauvegardé : /kaggle/working/cache_{MATCH}_reference.zip "
              f"({os.path.getsize(f'/kaggle/working/cache_{MATCH}_reference.zip') / 1e6:.0f} MB)")
    else:
        print("⚠️  Pas de cache de suivi à sauvegarder")


# ────────────────────────────────────────────────────────────

# ── Comparaison des BUTS avec la vérité (détection seule) + aperçu du rendu client ──
_VRAIS = [int(b.split(":")[0]) * 60 + int(b.split(":")[1]) for b in _M["buts"]]
_KO_DET = float((result or {}).get("kickoff_offset") or 0.0)
_fm = lambda t: f"{int(t)//60:02d}:{int(t)%60:02d}"
_ann = [(_g.get("time", 0) + _KO_DET, _g.get("goal_status", "—")) for _g in (result or {}).get("events", []) if _g.get("type") in ("goal", "score")]
print("\n" + "=" * 60 + "\nCOMPARAISON AVEC LES BUTS RÉELS\n" + "=" * 60)
print(f"  Coup d'envoi retenu : {_fm(_KO_DET)} (connu : {_fm(_M['coup_envoi_s'])})")
for _v in _VRAIS:
    _m = [(t, st) for t, st in _ann if abs(t - _v) <= 20]
    print(f"  but réel {_fm(_v)} : " + (f"✅ trouvé à {_fm(_m[0][0])} — {_m[0][1]}" if _m else "❌ MANQUÉ"))
_fp = [(t, st) for t, st in _ann if not any(abs(t - _v) <= 20 for _v in _VRAIS)]
for t, st in _fp:
    print(f"  but annoncé {_fm(t)} sans but réel — {st}")
_bm = (result or {}).get("bornes_mi_temps")
if _bm and _bm.get("segments"):
    print(f"  Bornes : KO1 {_fm(_bm['ko1'])} | Fin1MT {_fm(_bm['fin1mt'])} | KO2 {_fm(_bm['ko2'])} | FinMatch {_fm(_bm['finmatch'])}")
_tp = sum(1 for _v in _VRAIS if any(abs(t - _v) <= 20 for t, _ in _ann))
print(f"  BILAN DÉTECTION : {_tp}/{len(_VRAIS)} vrais buts | {len(_fp)} faux positifs dont {sum(st == 'confirmé' for _, st in _fp)} « confirmé(s) »")
try:
    from export.pdf import generate_pdf as _gen_pdf
    _gen_pdf(result, f"/kaggle/working/apercu_{MATCH}.pdf", sport="football")
    print(f"  Aperçu PDF : /kaggle/working/apercu_{MATCH}.pdf")
except Exception as _e_ap:
    print(f"  ⚠️ Aperçu PDF impossible : {type(_e_ap).__name__} : {_e_ap}")
try:
    import ai.gemini_validator as _gv_cache
    _gv_cache.gemini_cache_summary()
except Exception as _e_gc:
    print(f"  [GEMINI CACHE] bilan indisponible : {_e_gc}")


# ────────────────────────────────────────────────────────────


# ── relais vers la cellule 2/2 ──
import json as _json_be
_bm = (result or {}).get("bornes_mi_temps") or {}
_bornes = None
if all(_bm.get(k) is not None for k in ("ko1", "fin1mt", "ko2", "finmatch")):
    _bornes = [[float(_bm["ko1"]), float(_bm["fin1mt"])], [float(_bm["ko2"]), float(_bm["finmatch"])]]
_buts = [{"time_abs": float(b["time_abs"]), "statut": b.get("status") or b.get("goal_status") or b.get("statut") or "?",
          "equipe": b.get("tenue_equipe"), "niveau_equipe": b.get("niveau_equipe")}
         for b in (result or {}).get("buts", []) if b.get("time_abs") is not None]
if not _buts:   # repli : les événements « goal » du résultat
    _buts = [{"time_abs": float(g.get("time", 0)) + _KO_DET, "statut": g.get("goal_status", "?")}
             for g in (result or {}).get("events", []) if g.get("type") in ("goal", "score")]
_cache_be = f"/kaggle/working/ProjectAIMika/{OUT_DIR}/cache"
_json_be.dump({"match": MATCH, "video": VIDEO, "cache": _cache_be, "actions_abs": [], "bornes": _bornes, "buts": _buts,
               "t0": T0_BOUT_EN_BOUT, "duree_analyse_min": round((time.time() - T0_BOUT_EN_BOUT) / 60, 1), "joueur_seul": True},
              open(f"/kaggle/working/bout_en_bout_{MATCH}.json", "w"))
print(f"\n══ JOUEUR SEUL, ÉTAPE 1 TERMINÉE : {(time.time() - T0_BOUT_EN_BOUT) / 60:.0f} min | bornes : "
      + (" ; ".join(f"{_fm(a)}-{_fm(b)}" for a, b in _bornes) if _bornes else "NON TROUVÉES (tout le fichier sera lu)")
      + f" | {len(_buts)} but(s) : " + ", ".join(f"{_fm(b['time_abs'])} ({b['statut']})" for b in _buts))

# ── PROD : attribution, livrables et COÛT RÉEL de l'analyse (réponses Gemini écrites pendant ce run) ──
import json as _jp, collections as _cp
DUREE_ANALYSE_MIN = (time.time() - T0_BOUT_EN_BOUT) / 60
_att = (result or {}).get("attribution_equipes") or {}
if _att:
    print(f"   ÉQUIPES : {_att.get('tenues')} | score confirmé {_att.get('score_confirme')} | estimé {_att.get('score_estime')} | "
          f"sans équipe {_att.get('buts_sans_equipe')} | recompressées {_att.get('requetes_recompressees')} | "
          f"échecs {_att.get('questions_en_echec')}")
else:
    print("   ⚠️ ÉQUIPES : aucune attribution dans le résultat (module absent ou erreur : voir « [ATTRIBUTION] »)")
_rapport = f"/kaggle/working/ProjectAIMika/{OUT_DIR}/rapport.pdf"
if os.path.exists(_rapport):
    shutil.copy(_rapport, f"/kaggle/working/rapport_{MATCH}.pdf")
    print(f"   RAPPORT CLIENT : /kaggle/working/rapport_{MATCH}.pdf")
COUT_ANALYSE = _cp.defaultdict(lambda: [0, 0.0])
_out = os.environ.get("SCOUTIA_GEMINI_CACHE_OUT")
if _out and os.path.exists(_out):
    for _l in open(_out, encoding="utf-8"):
        try:
            _x = _jp.loads(_l)
            _ts = _dt_prod.datetime.fromisoformat(str(_x.get("enregistre_le") or "")[:19]).timestamp()
        except Exception:
            continue
        if _ts < _T_DEBUT_GEMINI - 120:
            continue
        _u = _x.get("usage") or {}
        _c = (int(_u.get("prompt_token_count") or 0) / 1e6 * 0.75
              + (int(_u.get("candidates_token_count") or 0) + int(_u.get("thoughts_token_count") or 0)) / 1e6 * 3.75)
        _f = (_x.get("contexte") or {}).get("fonction", "?")
        COUT_ANALYSE[_f][0] += 1; COUT_ANALYSE[_f][1] += _c
print(f"\n══ COÛT GEMINI DE L'ANALYSE ══")
for _f, (_n, _c) in sorted(COUT_ANALYSE.items(), key=lambda kv: -kv[1][1]):
    print(f"   {_f:34s} {_n:4d} question(s) ≈ {_c:.3f} $")
print(f"   TOTAL analyse ≈ {sum(v[1] for v in COUT_ANALYSE.values()):.2f} $ ({sum(v[0] for v in COUT_ANALYSE.values())} questions)")
print("   → ÉTAPE 2 : montage du joueur (enchaîné automatiquement)")




# ════ ÉTAPE 2 — MONTAGE DU JOUEUR (kaggle_joueur_v3_2.py) ════
# ════════════════════════════════════════════════════════════════════════════════════════════
# ScoutIA — MONTAGE JOUEUR v3.2 : PARSeq (v2) + CONFIRMATION GEMINI des segments lus une seule fois (07/11/2026)
# Constat du test n°16 : le modèle repère souvent le bon numéro UNE fois dans un segment où le joueur est bien là
# (règle : ≥ 2 lectures) — surtout sur les maillots pâles à numéros foncés. Ces segments, et seulement eux, sont relus
# par Gemini (4 vignettes, ≈ 0,007 $ chacun) ; la règle de retenue (≥ 2 lectures, aucun autre numéro) est inchangée.
# ─ base : test n°16 ─────────────────────────────────────────────────────────────────────────
# ScoutIA — TEST n°16 bleu ciel : modèle v1 contre v2 sur le résumé de Minerois Ster (07/11/2026)
# Réglages du test : NUMERO 16, COULEUR « bleu clair », FENETRES = les 21 actions du résumé, MONTAGE « presence »,
# PARSEQ_VERSION à mettre à "v1" puis à "v2". Vérité : participations à 81:51, 87:16, 89:53, 110:08.
# ─ base : v3.1 ─────────────────────────────────────────────────────────────────────────────
# ScoutIA — MONTAGE D'UN JOUEUR, v3.1 : TOUT LE MATCH + TOUCHES DE BALLE du joueur (06/11/2026)
# Nouveau v3.1 : FENETRES vide = tout le match est lu (gratuit avec PARSeq) ; les événements du suivi (possession,
# passe, dribble, tir…) attribués à une piste du joueur PENDANT une portion où son numéro est lu sont listés comme
# « touches », et MONTAGE = "touches" monte un extrait autour de chacune (sinon "presence" = v2.8).
# Réserve : ces événements viennent du suivi du ballon, peu fiable avec cette caméra — À VÉRIFIER dans la vidéo.
# ─ v3.0 ─────────────────────────────────────────────────────────────────────────────────────
# ScoutIA — MONTAGE D'UN JOUEUR, v3.0 : résumé + repère, numéros lus par NOTRE modèle PARSeq affiné (05/11/2026)
# Nouveau v3.0 : la lecture des numéros n'utilise plus Gemini mais parseq_scoutia_maillots_v1.pt (PARSeq © Darwin
# Bautista, Apache 2.0, affiné sur les lectures Gemini de Franchimont). Gratuit, donc AUCUN plafond. Seuil 0,8.
# Les couleurs des équipes viennent toujours de Gemini (2 questions). Lectures dans lectures_v30_<match>.json
# (jamais mélangées avec celles de Gemini). Tout le reste (segments, règle de retenue, raccords, repère) = v2.9.
# ─ v2.9 ─────────────────────────────────────────────────────────────────────────────────────
# Nouveau v2.9 : FENETRES = heures des actions du résumé. Seuls les segments du suivi qui touchent ces fenêtres sont lus
# par Gemini (coût réduit), et le montage contient TOUTES les fenêtres (le résumé complet), avec le repère dessiné
# là où le joueur est reconnu. Reste identique à la v2.8 (même lecture, mêmes raccords, même repère).
# ─ v2.8 ─────────────────────────────────────────────────────────────────────────────────────
# Ne modifie NI le suivi NI la détection des buts. CPU ; Gemini : ~1 question par segment lu (plafonné).
#
# Base : v2.7 (lecture en réflexion limitée, contrôle des buts). Avec CONSENSUS_TEMPORAL = False (défaut), le
# montage est construit EXACTEMENT comme en v2.7 (seuls changent les noms de fichiers et une colonne « consensus »
# à 0 dans les CSV).
#
# Nouveau v2.8 (option CONSENSUS_TEMPORAL = True) : CONSENSUS DANS LE TEMPS, testé le 29/10 sur Franchimont
# (+79 s de présence du 10, dégagement raté de 49:37 retrouvé, 15 groupes vérifiés à l'œil).
#  - Une lecture « 10 » isolée est acceptée si d'autres lectures « 10 » existent à moins de CONS_S s, au même
#    endroit de l'image ; le groupe doit avoir ≥ CONS_MIN_LECTURES lectures dont ≥ CONS_MIN_ROUGES sur des
#    vignettes ROUGES ; aucune lecture sur une vignette de l'équipe adverse (elle a aussi un n° 10).
#  - Lectures simultanées à deux endroits (≤ 0,2 s, > 0,5 hauteur) : seule la lecture la MOINS soutenue par le
#    reste du groupe est exclue (les deux si égalité), puis le groupe est revalidé.
#  - Groupes faibles (pas assez de lectures rouges après exclusion) : jamais ajoutés, mais tracés.
#  - Sources : lectures normales + lectures renforcées (lectures_renforcees_<match>.json en entrée, lecture seule).
#  - Présence : ± GARDE_S autour de chaque lecture gardée, sans franchir une vignette de l'équipe adverse ou d'un
#    autre numéro ; ajoutée seulement là où le joueur n'est pas déjà repéré (pas de doublon).
#  - Montage : repère ORANGE « #10* » pour le consensus (jaune = lu, bleu = raccord). Trace :
#    consensus_v28_*.csv et verification_consensus_v30_p*.jpg.
# Validé seulement sur le cache en mode « filtre » : à reconfirmer sur le run complet en mode « historique ».
#
# MODES : "diagnostic" (cache seul, gratuit) | "essai" (pistes TIDS_DIAG seulement ; raccords cherchés dans tout
#         le cache) | "complet".
# ════════════════════════════════════════════════════════════════════════════════════════════
# ── RÉGLAGES ──────────────────────────────────────────────────────────────────────────────────
MODE         = "complet"      # "diagnostic", "essai" ou "complet"
MATCH        = "minerois_ster"
VIDEO        = "/kaggle/input/datasets/michaelziant/mineroisster-complet/minerois-ster-2025-11-09.mp4"
CACHE        = "/kaggle/input/datasets/michaelziant/codeprojectai/cache_minerois_ster_reference"
                              # dossier ou archive du cache du suivi ; vide = recherche automatique
NUMERO       = 16
COULEUR      = "bleu clair"   # rouge, bleu, bleu clair, blanc, noir, jaune, vert, orange, violet, gris
PARSEQ_VERSION = "v2"     # ← TEST n°16 : lancer une fois avec "v1", une fois avec "v2" ("" = le plus récent disponible)
EQUIPE       = None           # None = équipe choisie par la couleur ; 0 ou 1 = équipe imposée
MARGE_AVANT  = 4.0            # secondes de contexte avant une portion retenue
MARGE_APRES  = 3.0            # secondes après
FUSION_S     = 8.0            # deux portions à moins de 8 s = même séquence
MIN_DUREE_S  = 2.0            # segment lu s'il dure au moins 2 s
MIN_HAUTEUR  = 90             # hauteur minimale du joueur (px) pour qu'un numéro soit lisible
N_LECTURES   = 12             # 09/11 : 12 avec la LECTURE RAPIDE (le temps ne dépend presque plus du nombre d images : la vidéo est décodée une fois)
MIN_CONCORDANTES = 2          # le numéro doit être lu au moins 2 fois sur des vignettes de la bonne couleur
LECTURE_UNIQUE_SI_COULEUR = False   # True = accepter UNE lecture si ≥ 2 vignettes consécutives de la bonne
                                    # couleur et aucun autre numéro lu (hypothèse à tester, pas validée)
MAX_SEGMENTS = 0              # segments EXAMINÉS en mode complet (couleur mesurée, sans Gemini) ; 0 = tous
MAX_QUESTIONS = 10**9           # plafond de NOUVELLES questions Gemini par run (lectures réutilisées non comptées)
TENUE_REPERE_S = 0.5          # repère tenu au plus 0,5 s après la dernière position réelle
# v2.9 : résumé du match (heures des actions repérées par le pipeline) ; [] = comportement v2.8 (tout le match)
FENETRES = ["15:21", "18:27", "23:28", "25:12", "30:23", "42:48", "50:15", "55:42", "74:51", "81:51", "87:16",
            "89:53", "90:57", "92:38", "96:29", "99:02", "103:02", "106:53", "110:08", "111:35", "119:50"]   # test : le résumé
MONTAGE = "touches" if True else "presence"  # v3.1 : "touches" = un extrait autour de chaque touche de balle ; "presence" = comportement v2.8
TOUCHE_AVANT, TOUCHE_APRES = 4.0, 3.0
TYPES_TOUCHES = {"possession", "pass", "long_pass", "dribble", "shot", "interception", "progressive_run", "fast_break", "goal"}
FENETRE_AVANT, FENETRE_APRES = 10.0, 15.0
# ── BOUT EN BOUT (07/11) : si la cellule 1/2 a tourné dans ce notebook, ses résultats remplacent les réglages ──
import os, glob as _gbe, json as _jbe, time as _tbe
T_DEBUT_MONTAGE = _tbe.time()
_be = next(iter(sorted(_gbe.glob("/kaggle/working/bout_en_bout_*.json"))
               + sorted(_gbe.glob("/kaggle/input/**/bout_en_bout_*.json", recursive=True))), None)   # 08/11 : aussi en entrée
BOUT_EN_BOUT = None
# 08/11 : relais de l'étape 1 COLLÉ ICI (utilisé si aucun fichier bout_en_bout_*.json n'est trouvé). Mettre None pour l'ignorer.
RELAIS_MANUEL_ANCIEN = {"match": "minerois_ster", "video": "/kaggle/input/datasets/michaelziant/mineroisster-complet/minerois-ster-2025-11-09.mp4",
                 "cache": "/kaggle/working/ProjectAIMika/outputs/minerois_ster_reference/cache", "actions_abs": [],
                 "bornes": [[723.0, 3445.0], [4271.0, 7227.0]],
                 "buts": [{"time_abs": 3341.0, "statut": "?"}, {"time_abs": 5236.0, "statut": "?"}, {"time_abs": 5961.0, "statut": "?"},
                          {"time_abs": 6183.0, "statut": "?"}, {"time_abs": 6612.0, "statut": "?"}, {"time_abs": 7190.0, "statut": "?"}],
                 "t0": None, "duree_analyse_min": 100.9, "joueur_seul": True}
RELAIS_MANUEL = None   # run de production : le relais vient de l'étape 1 de cette session
if _be or RELAIS_MANUEL:
    BOUT_EN_BOUT = _jbe.load(open(_be, encoding="utf-8")) if _be else dict(RELAIS_MANUEL)
    print(f"   relais : {'fichier ' + _be if _be else 'RELAIS_MANUEL (collé dans le script)'}")
    MATCH, VIDEO, CACHE = BOUT_EN_BOUT["match"], BOUT_EN_BOUT["video"], BOUT_EN_BOUT["cache"]
    if not os.path.exists(os.path.join(CACHE, "frames_data.pkl")):
        # 08/11 : cellule 2 lancée dans un AUTRE notebook (sortie de la cellule 1 en entrée) : on décompresse le cache neuf
        import zipfile as _zbe
        # ordre : dossier « cache_<match>_neuf » (dataset déjà décompressé), puis zip « _neuf » ou « _reference » de la
        # SORTIE de l'étape 1. JAMAIS l'ancien dossier codeprojectai/cache_<match>_reference (autre suivi).
        _tous = sorted(_gbe.glob("/kaggle/input/**/frames_data.pkl", recursive=True))
        print("   caches de suivi présents dans les entrées : " + (" | ".join(_tous) if _tous else "aucun"))
        # Kaggle peut décompresser le zip de la sortie de l'étape 1 en un dossier « cache_<match>_reference » : on l'accepte
        # S'IL N'EST PAS dans codeprojectai (là se trouve l'ancien suivi, du même nom).
        def _date_cache(f):
            try:
                m = _jbe.load(open(os.path.join(os.path.dirname(f), "meta.json"), encoding="utf-8"))
                return str(m.get("saved_at") or m.get("date") or "")
            except Exception:
                return ""
        _t0_relais = BOUT_EN_BOUT.get("t0")
        _jour_relais = _tbe.strftime("%Y-%m-%d", _tbe.gmtime(_t0_relais)) if _t0_relais else "2026-11-07"
        for f in _tous:
            if f"cache_{MATCH}" in f:
                print(f"     {f} : sauvegardé le {_date_cache(f) or '?'} (étape 1 du {_jour_relais})")
        # nouveau cache = nommé « _neuf », OU hors codeprojectai, OU (même dans codeprojectai) sauvegardé le jour de l'étape 1
        # ou après : l'utilisateur a pu remplacer l'ancien cache par le nouveau sous le même nom.
        _dos = next(iter([f for f in _tous if f"cache_{MATCH}_neuf" in f]
                         + [f for f in _tous if f"cache_{MATCH}_reference" in f and "/codeprojectai/" not in f]
                         + [f for f in _tous if f"cache_{MATCH}_reference" in f and _date_cache(f)[:10] >= _jour_relais]), None)
        _z = next(iter(sorted(_gbe.glob(f"/kaggle/input/**/cache_{MATCH}_neuf.zip", recursive=True))
                       + sorted(_gbe.glob(f"/kaggle/input/**/cache_{MATCH}_reference.zip", recursive=True))), None)
        if _dos:
            CACHE = os.path.dirname(_dos)
        elif _z:
            _dest = f"/kaggle/working/cache_{MATCH}_neuf"
            if not os.path.exists(os.path.join(_dest, "frames_data.pkl")):
                _zbe.ZipFile(_z).extractall(_dest)
            _fd = next(iter(_gbe.glob(f"{_dest}/**/frames_data.pkl", recursive=True)), None)
            if not _fd:
                raise SystemExit(f"❌ frames_data.pkl absent de {_z}")
            CACHE = os.path.dirname(_fd)
        else:
            raise SystemExit(f"❌ cache NEUF introuvable : ajouter en entrée la sortie de l'étape 1 (cache_{MATCH}_reference.zip), "
                             f"ou un dataset « cache_{MATCH}_neuf » — l'ancien cache de codeprojectai n'est volontairement pas utilisé")
        print(f"   cache NEUF utilisé : {CACHE}")
    FENETRES = [f"{int(t) // 60:02d}:{int(t) % 60:02d}" for t in BOUT_EN_BOUT["actions_abs"]]
    print(f"   BOUT EN BOUT : {MATCH} | cache neuf {CACHE} | {len(FENETRES)} action(s) du résumé produites par le pipeline")
else:
    print("   ⚠️⚠️ AUCUN RELAIS bout_en_bout_*.json TROUVÉ (ni dans /kaggle/working, ni dans les entrées) : réglages PAR DÉFAUT "
          "(ancien cache de codeprojectai, résumé de 21 actions). Pour le test « joueur seul », ajouter en entrée la SORTIE "
          "du notebook de la cellule 1 (Add Input → Notebook Output Files).")
# découpage des pistes (hypothèses à valider)
COUPURE_TROU_S   = 1.0        # trou plus long → nouveau segment
SAUT_BASE_H      = 1.0        # déplacement toléré entre deux positions : SAUT_BASE_H + SAUT_H_PAR_S × durée,
SAUT_H_PAR_S     = 6.0        #   en hauteurs de joueur (≈ 6 hauteurs/s = sprint)
RAPPORT_TAILLE_MAX = 1.6      # changement de hauteur du cadre plus brusque → nouveau segment
EXIGER_TSU   = False          # True = arrêter si le cache n'a pas time_since_update
PREFILTRE_EQUIPE = 0.0        # 07/11 : 0 (était 0,1). Le cache étiquette mal l équipe bleu ciel (souvent « None ») : des morceaux du n°16 de 150 à 245 px (de près !) étaient ignorés. La couleur mesurée vignette par vignette protège toujours l identité.
                              # (champ peu fiable : simple tri pour limiter le coût ; 0 = pas de tri)
SEUIL_COULEUR = 0.15          # part minimale des pixels du haut du maillot dans la couleur de l'équipe
TIDS_DIAG    = ["2383", "6824", "7969"]   # pistes examinées (diagnostic) ou lues (essai)
FICHIER_LECTURES = ""         # vide = /kaggle/working/lectures_v30_<match>.json ; fusionne v2.8 à v2.2
# raccordement (hypothèses à valider)
RACCORD_MAX_S    = 2.0        # écart maximal entre la fin d'une portion et le début du segment voisin
RACCORD_RECOUV_S = 1.0        # chevauchement toléré (deux pistes sur le même joueur au même moment)
RACCORD_BASE_H   = 1.0        # distance tolérée au raccord : RACCORD_BASE_H + RACCORD_H_PAR_S × écart,
RACCORD_H_PAR_S  = 3.0        #   en hauteurs de joueur (la caméra Veo bouge : distance dans l'image, pas sur le terrain)
RACCORD_MAX_CHAINE = 4        # raccords successifs au plus, dans chaque sens
RACCORD_HMIN     = 40         # hauteur minimale (px) d'une vignette de couleur pour un raccord
MARQUER_RACCORDS = True       # repère bleu « #10? » sur les raccords (vérification) ; False = jaune partout
# variantes de raccord (v2.5) : calculées et comparées ; le montage utilise RACCORD_MONTAGE
RACCORD_RAPPORT  = 2.0        # B : le plus proche est accepté si le suivant est au moins 2 fois plus loin
RACCORD_RECOUV_MAX_H = 0.3    # v2.6 : pistes qui se chevauchent dans le temps → même joueur seulement si ≤ 0,3 hauteur
EXTENSION_MAX_S  = 2.0        # v2.6 : prolongement maximal d'une portion lue au-delà des vignettes lues
RACCORD_MONTAGE  = "B"        # "ref" ou "B" (« B partiel (v2.4) » : comparaison seulement, jamais monté)
LIGNES_PAR_PAGE  = 6          # 07/11 : 6 (était 20). Avec 12 vignettes par ligne, les pages plus hautes étaient réduites à l envoi
MODELE       = "gemini-3.8-flash"
REFLEXION_LECTURE = "low"     # v2.7 : réflexion limitée pour la lecture des numéros ; None = réflexion normale
CONTROLE_BUT_AVANT_S = 20     # v2.7 : contrôle des buts, fenêtre avant le but (s)
# v2.8 : consensus dans le temps (désactivé par défaut : comportement v2.7 inchangé)
CONSENSUS_TEMPORAL = True     # v2.9 : activé pour étendre le repère (sans nouvelle question Gemini)
CONS_S, CONS_BASE_H, CONS_H_PAR_S = 2.0, 1.0, 3.0   # lectures liées : ≤ 2 s, distance ≤ 1 + 3 × écart (hauteurs)
CONS_MIN_LECTURES, CONS_MIN_ROUGES = 2, 2           # groupe valable : ≥ 2 lectures « 10 », dont ≥ 2 sur vignettes rouges
GARDE_S = 1.0                                       # présence gardée ± 1 s autour de chaque lecture
# coût (29/10) : tarif de la passation pour gemini-3.8-flash, en dollars par million de jetons. Les jetons de
# RÉFLEXION sont comptés au prix de la sortie (règle de facturation Google à revérifier sur la page des tarifs).
PRIX_ENTREE_M = 0.75
PRIX_SORTIE_M = 3.75
TAUX_EUR      = 0.92          # conversion approximative, pour l'affichage seulement

import os, glob, json, pickle, zipfile, subprocess, collections, csv, time
import numpy as np
import cv2

T0 = time.time()
SORTIE = "/kaggle/working"
fmt = lambda t: f"{int(t)//60:02d}:{int(t)%60:02d}"
dans_image = lambda b: 0 <= b[0] < b[2] <= 1920 and 0 <= b[1] < b[3] <= 1080

if MODE not in ("diagnostic", "essai", "complet"):
    raise SystemExit(f"❌ MODE = {MODE!r} : choisis « diagnostic », « essai » ou « complet »")


def _cle_gemini():
    """Clé lue hors du code : variable d'environnement, sinon secret Kaggle « GEMINI_API_KEY »
    (menu Add-ons > Secrets du notebook)."""
    cle = os.getenv("GEMINI_API_KEY", "")
    if not cle:                               # v2.9 : comme les autres scripts, param.env du jeu « mziparameter »
        import glob as _g
        pe = next(iter(_g.glob("/kaggle/input/**/mziparameter/**/param.env", recursive=True)), None)
        if pe:
            for l in open(pe, encoding="utf-8"):
                if l.strip().startswith("GEMINI_API_KEY="):
                    cle = l.split("=", 1)[1].strip().strip('"').strip("'")
    if not cle:
        try:
            from kaggle_secrets import UserSecretsClient
            cle = UserSecretsClient().get_secret("GEMINI_API_KEY") or ""
        except Exception:
            cle = ""
    return cle.strip()


GEMINI_KEY = _cle_gemini() if MODE != "diagnostic" else ""

# ── MESURE DU COÛT GEMINI (jetons renvoyés par l'API après chaque question) ──────────────────────
import threading as _th
COUT = {"questions": 0, "entree": 0, "sortie": 0, "reflexion": 0, "sans_mesure": 0}
COUT_PAR_USAGE = collections.defaultdict(lambda: {"questions": 0, "entree": 0, "sortie": 0, "reflexion": 0})
_cout_verrou = _th.Lock()


def compter(reponse, usage):
    """Ajoute les jetons d'une réponse Gemini (entrée, sortie, réflexion) aux totaux du run."""
    um = getattr(reponse, "usage_metadata", None)
    e = int(getattr(um, "prompt_token_count", 0) or 0)
    so = int(getattr(um, "candidates_token_count", 0) or 0)
    re_ = int(getattr(um, "thoughts_token_count", 0) or 0)
    with _cout_verrou:
        COUT["questions"] += 1
        if um is None:
            COUT["sans_mesure"] += 1
        for cle, v in (("entree", e), ("sortie", so), ("reflexion", re_)):
            COUT[cle] += v; COUT_PAR_USAGE[usage][cle] += v
        COUT_PAR_USAGE[usage]["questions"] += 1


def dollars(c):
    return c["entree"] / 1e6 * PRIX_ENTREE_M + (c["sortie"] + c["reflexion"]) / 1e6 * PRIX_SORTIE_M

# ── 1. CACHE DU SUIVI ─────────────────────────────────────────────────────────────────────────
if MODE != "diagnostic" and not os.path.exists(VIDEO):
    raise SystemExit(f"❌ Vidéo introuvable : {VIDEO}")
# Le cache peut être une archive .zip, ou un dossier (Kaggle DÉCOMPRESSE les .zip ajoutés à un dataset) :
# dans ce cas, le bon dossier est reconnu par son meta.json (chemin de la vidéo du match).
DOSSIER = None


def _dossier_cache(chemin):
    """Dossier contenant meta.json, events.pkl, jersey_map.pkl et frames_data.pkl, à partir d'un dossier (même
    imbriqué) ou d'une archive (même sans l'extension .zip)."""
    if os.path.isfile(chemin) and zipfile.is_zipfile(chemin):
        cible = f"{SORTIE}/cache_{MATCH}"
        with zipfile.ZipFile(chemin) as z:
            z.extractall(cible)
        chemin = cible
    if os.path.isdir(chemin):
        for m in sorted(glob.glob(os.path.join(chemin, "**", "meta.json"), recursive=True)):
            d = os.path.dirname(m)
            if all(os.path.exists(os.path.join(d, f)) for f in ("events.pkl", "jersey_map.pkl", "frames_data.pkl")):
                return d
    return None


if CACHE:
    DOSSIER = _dossier_cache(CACHE)
    if not DOSSIER:
        contenu = sorted(os.listdir(CACHE))[:20] if os.path.isdir(CACHE) else ("fichier" if os.path.exists(CACHE) else "INEXISTANT")
        raise SystemExit(f"❌ CACHE = {CACHE} : aucun cache du suivi lisible (contenu : {contenu})")
    print(f"✅ Cache : {DOSSIER}")
zips = [] if DOSSIER else sorted(glob.glob(f"/kaggle/input/**/cache_{MATCH}_reference*.zip", recursive=True))
if DOSSIER:
    pass
elif zips:
    DOSSIER = f"{SORTIE}/cache_{MATCH}"
    with zipfile.ZipFile(zips[-1]) as z:
        z.extractall(DOSSIER)
    print(f"✅ Cache (archive) : {zips[-1]}")
else:
    candidats_c = []
    for m in glob.glob("/kaggle/input/**/meta.json", recursive=True):
        d = os.path.dirname(m)
        if not all(os.path.exists(os.path.join(d, f)) for f in ("events.pkl", "jersey_map.pkl", "frames_data.pkl")):
            continue
        try:
            video_meta = str(json.load(open(m)).get("video_path", "")).lower()
        except Exception:
            video_meta = ""
        candidats_c.append((d, video_meta))
        if MATCH.lower() in video_meta or MATCH.lower() in d.lower():
            DOSSIER = d
    if DOSSIER:
        print(f"✅ Cache (dossier décompressé) : {DOSSIER}")
    else:
        print("   caches trouvés en entrée :", [(c[0], os.path.basename(c[1])) for c in candidats_c] or "aucun")
        raise SystemExit(f"❌ Cache du suivi de « {MATCH} » introuvable — ajoute en entrée le dataset qui contient "
                         f"cache_{MATCH}_reference (archive ou dossier avec meta.json, events.pkl, jersey_map.pkl, frames_data.pkl)")


class _S:
    def __init__(self, *a, **k): pass

    def __setstate__(self, st):
        if isinstance(st, dict): self.__dict__.update(st)


class _U(pickle.Unpickler):
    def find_class(self, m, n):
        try: return super().find_class(m, n)
        except Exception: return _S


charger = lambda nom: _U(open(os.path.join(DOSSIER, nom), "rb")).load()
meta = json.load(open(os.path.join(DOSSIER, "meta.json")))
KO, FPS = float(meta["kickoff_offset"]), float(meta["fps"])
jersey_map = {str(k): v for k, v in (charger("jersey_map.pkl") or {}).items()}
events = charger("events.pkl")
frames = charger("frames_data.pkl")
print(f"   coup d'envoi {fmt(KO)} | {len(events)} événements | {len(frames)} images suivies | {len(jersey_map)} numéros lus")
# empreinte du cache : les lectures sauvegardées ne sont réutilisées que pour CE cache
SIGNATURE = f"{os.path.basename(str(meta.get('video_path', '')))}|{KO:.3f}|{FPS:.3f}|{len(frames)}"



# ── POSITIONS ET SEGMENTS (CPU, aucun appel) ──────────────────────────────────────────────────
def _tsu(p):
    v = p.get("time_since_update")
    try:
        return None if v is None else int(v)
    except Exception:
        return None


N_POS = N_TSU = N_PREDITES = N_HORS = N_SANS_TID = 0
for f in frames:
    for p in f.get("players", []):
        N_POS += 1
        if _tsu(p) is not None:
            N_TSU += 1
AVEC_TSU = N_TSU > 0
if AVEC_TSU:
    print(f"✅ time_since_update présent ({100 * N_TSU / max(1, N_POS):.1f} % des positions) : "
          f"seules les vraies détections (== 0) sont utilisées")
else:
    msg = ("time_since_update ABSENT de ce cache (cache antérieur au replay.py du 28/10) : les positions PRÉDITES "
           "par le suivi ne peuvent pas être écartées. En mode « historique », le résultat serait peu sûr.")
    if EXIGER_TSU:
        raise SystemExit("❌ " + msg)
    print("⚠️ " + msg)

pistes = collections.defaultdict(dict)           # tracker_id -> {image suivie: (cadre, équipe du cache)}
for f in frames:
    num = int(f.get("frame", 0))
    for p in f.get("players", []):
        tid, b = p.get("tracker_id"), p.get("bbox")
        if tid is None:
            N_SANS_TID += 1; continue
        if AVEC_TSU and _tsu(p) != 0:
            N_PREDITES += 1; continue
        if b is None or not dans_image(b):
            N_HORS += 1; continue
        if num in pistes[str(tid)]:
            continue
        pistes[str(tid)][num] = ([float(v) for v in b], p.get("team"))
if not pistes:
    raise SystemExit("❌ Ce cache ne contient pas l'identifiant DeepSort (tracker_id) : refais le suivi avec la version actuelle")


def decouper(pos):
    """Coupe une piste aux ruptures probables d'identité. Renvoie [(liste de numéros d'image, raison de la
    coupure qui PRÉCÈDE ce segment ou None)]."""
    nums = sorted(pos)
    segs, courant, raison = [], [nums[0]], None
    for a, b in zip(nums, nums[1:]):
        dt = (b - a) / FPS
        ba, bb = pos[a][0], pos[b][0]
        ha, hb = max(1.0, ba[3] - ba[1]), max(1.0, bb[3] - bb[1])
        d = np.hypot((ba[0] + ba[2]) / 2 - (bb[0] + bb[2]) / 2, ba[3] - bb[3]) / ((ha + hb) / 2)
        r = None
        if dt > COUPURE_TROU_S:
            r = f"trou {dt:.1f} s"
        elif d > SAUT_BASE_H + SAUT_H_PAR_S * dt:
            r = f"saut {d:.1f} h en {dt:.2f} s"
        elif max(hb / ha, ha / hb) > RAPPORT_TAILLE_MAX:
            r = f"taille ×{max(hb / ha, ha / hb):.1f}"
        if r:
            segs.append((courant, raison)); courant, raison = [b], r
        else:
            courant.append(b)
    segs.append((courant, raison))
    return segs


segments = {}                                    # « tid@début-fin » -> {"tid", "pos", "raison"}
raisons = collections.Counter()
for tid, pos in pistes.items():
    for nums, raison in decouper(pos):
        sid = f"{tid}@{nums[0]}-{nums[-1]}"
        segments[sid] = {"tid": tid, "pos": {n: pos[n] for n in nums}, "raison": raison}
        if raison:
            raisons[raison.split()[0]] += 1
print(f"   {len(pistes)} pistes DeepSort → {len(segments)} segments | coupures : {dict(raisons)}"
      + (f" | positions prédites écartées : {N_PREDITES}" if AVEC_TSU else "")
      + f" | hors image : {N_HORS}" + (f" | sans tracker_id : {N_SANS_TID}" if N_SANS_TID else ""))

debut = lambda sid: KO + min(segments[sid]["pos"]) / FPS
fin = lambda sid: KO + max(segments[sid]["pos"]) / FPS


# ── DIAGNOSTIC (MODE = "diagnostic") ──────────────────────────────────────────────────────────
def _court(v, n=90):
    s = repr(v)
    return s if len(s) <= n else s[:n] + "…"


def diagnostic():
    """Mesures seulement, aucune conclusion automatique."""
    print(f"\n{'─' * 90}\nDIAGNOSTIC DU CACHE (CPU, aucun appel Gemini, vidéo non lue)\n{'─' * 90}")
    champs = collections.Counter(k for f in frames for p in f.get("players", []) for k in p)
    print("1) champs d'une position :", ", ".join(f"{k} {n}" for k, n in champs.most_common()))
    nums = sorted({int(f.get("frame", 0)) for f in frames})
    ecarts = collections.Counter(np.diff(nums).tolist()) if len(nums) > 1 else collections.Counter()
    if ecarts:
        typique = ecarts.most_common(1)[0][0] / FPS
        print(f"2) écart typique entre images suivies : {typique:.2f} s → TENUE_REPERE_S = {TENUE_REPERE_S} s "
              + ("suffisante" if TENUE_REPERE_S >= 1.5 * typique else f"trop courte, prévoir ≥ {1.5 * typique:.2f} s"))
    tsu = collections.Counter(_tsu(p) for f in frames for p in f.get("players", []))
    print(f"3) {N_POS} positions ({N_POS / max(1, len(frames)):.1f} par image suivie) | time_since_update : "
          + (", ".join(f"{k}: {v}" for k, v in sorted(tsu.items(), key=lambda kv: (kv[0] is None, kv[0] or 0))[:8])
             if AVEC_TSU else "ABSENT"))
    fiable = collections.Counter(p.get("fiable") for f in frames[::50] for p in f.get("players", []))
    print(f"   « fiable » (échantillon) : {dict(fiable)} (non utilisé : même critère que time_since_update)")
    durees = np.array([(max(s["pos"]) - min(s["pos"])) / FPS for s in segments.values()])
    print(f"4) segments : {len(segments)} | durée médiane {np.median(durees):.1f} s | ≥ {MIN_DUREE_S} s : "
          f"{int((durees >= MIN_DUREE_S).sum())} | coupures : {dict(raisons)}")
    print(f"\n5) pistes examinées : {TIDS_DIAG}")
    for tid in map(str, TIDS_DIAG):
        sids = sorted((s for s, v in segments.items() if v["tid"] == tid), key=debut)
        if not sids:
            print(f"   piste {tid} : absente"); continue
        print(f"   piste {tid} : {len(sids)} segment(s)")
        for s in sids:
            v = segments[s]
            eq = collections.Counter(e for _, e in v["pos"].values())
            hmax = max(b[3] - b[1] for b, _ in v["pos"].values())
            print(f"     {fmt(debut(s))} → {fmt(fin(s))} ({fin(s) - debut(s):4.1f} s, {len(v['pos']):3d} pos., "
                  f"hauteur max {hmax:.0f} px) | équipe du cache {dict(eq)}"
                  + (f" | coupé avant : {v['raison']}" if v["raison"] else ""))
    print(f"\n✅ Diagnostic terminé en {time.time() - T0:.0f} s.")


if MODE == "diagnostic":
    diagnostic()
    raise SystemExit(0)

# ── LECTURES SAUVEGARDÉES : FUSION DE TOUS LES FICHIERS (format 2) ─────────────────────────────
FICHIER_LECTURES = FICHIER_LECTURES or f"{SORTIE}/lectures_v30_{MATCH}_{PARSEQ_VERSION or 'auto'}.json"
VERSION = "v2.8"


def fichiers_lectures():
    """Du plus récent au plus ancien : v2.7 à v2.2 ; pour chacun /kaggle/working puis /kaggle/input."""
    out = [FICHIER_LECTURES]
    for v in ("v30",):                              # v3.0 : uniquement les lectures PARSeq
        out += [f"{SORTIE}/lectures_{v}_{MATCH}.json"] + sorted(glob.glob(f"/kaggle/input/**/lectures_{v}_{MATCH}.json", recursive=True))
    return [c for c in dict.fromkeys(out) if os.path.exists(c)]


def _lue(e):
    return e.get("numeros") is not None and not e.get("non_lu")


def fusionner_lectures():
    lectures, coul_r, conflits, sources, sauve = {}, {}, [], [], None
    for c in fichiers_lectures():
        try:
            d = json.load(open(c, encoding="utf-8"))
        except Exception as e:
            print(f"   ⚠️ {c} illisible ({e}) : ignoré"); continue
        if d.get("format") != 2 or d.get("match") != MATCH or d.get("signature") != SIGNATURE or d.get("modele") != MODELE:
            print(f"   ⚠️ {c} : autre format, match, cache ou modèle : ignoré"); continue
        nom = os.path.basename(c)
        if sauve is None or (sauve.get("couleurs_source") != "gemini" and d.get("couleurs_source") == "gemini"):
            sauve = d                                    # couleurs des équipes : fichier le plus récent issu de Gemini
        n_ajout = 0
        for sid, e in (d.get("segments") or {}).items():
            e = dict(e); e.setdefault("source", nom)
            deja = lectures.get(sid)
            if deja is None:
                lectures[sid] = e; n_ajout += 1
            elif _lue(e) and not _lue(deja):
                lectures[sid] = e; n_ajout += 1          # une vraie lecture l'emporte sur « évitée » ou « plafond »
            elif _lue(e) and _lue(deja) and (e.get("numeros") != deja.get("numeros") or e.get("choix") != deja.get("choix")):
                conflits.append({"segment": sid, "gardee": deja.get("source"), "numeros_gardes": deja.get("numeros"),
                                 "ecartee": e.get("source"), "numeros_ecartes": e.get("numeros"),
                                 "meme_images": e.get("choix") == deja.get("choix")})
        for sid, e in (d.get("raccord_couleurs") or {}).items():
            coul_r.setdefault(sid, e)
        sources.append((nom, c, n_ajout))
    return lectures, coul_r, conflits, sources, sauve


lectures, COUL_R, CONFLITS, SOURCES_L, sauve = fusionner_lectures()
for nom, c, n in SOURCES_L:
    print(f"✅ Lectures : {c} → {n} segment(s) apportés")
if SOURCES_L:
    print(f"   fusion : {len(lectures)} segment(s) connus, dont {sum(1 for e in lectures.values() if _lue(e))} lus par Gemini | "
          f"conflits entre fichiers : {len(CONFLITS)} (le plus récent gardé, l'autre journalisé)")
with open(f"{SORTIE}/lectures_conflits_v28_{MATCH}.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["segment", "gardee", "numeros_gardes", "ecartee", "numeros_ecartes", "meme_images"], delimiter=";")
    w.writeheader(); w.writerows(CONFLITS)


def sauver_lectures():
    d = {"format": 2, "match": MATCH, "signature": SIGNATURE, "modele": MODELE,
         "couleurs": {str(k): v for k, v in COULEUR_EQUIPE.items()}, "couleurs_source": SOURCE_COULEURS,
         "segments": lectures, "raccord_couleurs": COUL_R, "conflits": CONFLITS}
    tmp = FICHIER_LECTURES + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(d, f, ensure_ascii=False)
    os.replace(tmp, FICHIER_LECTURES)


# ── 2. COULEUR DE CHAQUE ÉQUIPE ───────────────────────────────────────────────────────────────
def nom_couleur(h, s, v):
    """Teinte OpenCV (0-179), saturation et valeur (0-255) -> nom de couleur en français."""
    if v < 60:                    return "noir"
    if s < 45:                    return "blanc" if v > 165 else "gris"
    if h < 9 or h >= 170:         return "rouge"
    if h < 22:                    return "orange"
    if h < 34:                    return "jaune"
    if h < 85:                    return "vert"
    if h < 100:                   return "bleu clair"
    if h < 130:                   return "bleu"
    return "violet"


COULEURS_VALIDES = ["rouge", "bleu", "bleu clair", "blanc", "noir", "jaune", "vert", "orange", "violet", "gris"]


def couleurs_par_gemini():
    """Même source que la pré-analyse du site (Gemini, juste après le coup d'envoi), mais RATTACHÉE aux équipes du
    suivi : les joueurs de l'équipe 0 sont encadrés en cyan « A », ceux de l'équipe 1 en magenta « B »."""
    try:
        from google import genai
        from google.genai import types
    except Exception as e:
        print(f"   (Gemini indisponible : {e})"); return {}
    client = genai.Client(api_key=GEMINI_KEY)
    par_frame = {f.get("frame", 0): f for f in frames}
    cles = np.array(sorted(par_frame))
    votes = {0: collections.Counter(), 1: collections.Counter()}
    cap_g = cv2.VideoCapture(VIDEO)
    for dt in (10, 30, 60):
        num = int(cles[np.argmin(np.abs(cles - dt * FPS))])
        f = par_frame[num]
        cap_g.set(cv2.CAP_PROP_POS_MSEC, (KO + num / FPS) * 1000)
        ok, img = cap_g.read()
        if not ok:
            continue
        n = {0: 0, 1: 0}
        for p in f.get("players", []):
            b = p.get("bbox"); eq_ = p.get("team")
            # vraies détections seulement quand le cache le permet (time_since_update == 0) ; « conf » vaut toujours 1
            if eq_ not in (0, 1) or b is None or (AVEC_TSU and _tsu(p) != 0) or not dans_image(b):
                continue
            x1, y1, x2, y2 = [int(v) for v in b]
            c = (255, 255, 0) if eq_ == 0 else (255, 0, 255)                    # BGR : cyan / magenta
            cv2.rectangle(img, (x1, y1), (x2, y2), c, 3)
            cv2.putText(img, "A" if eq_ == 0 else "B", (x1, max(15, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, c, 3)
            n[eq_] += 1
        if min(n.values()) < 2:
            print(f"   Gemini, coup d'envoi + {dt} s : image ignorée (moins de 2 joueurs encadrés par équipe : {n})")
            continue
        cv2.imwrite(f"{SORTIE}/equipes_ko_plus_{dt}s.jpg", img)
        _, jpg = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 85])
        consigne = ("Football match image. Players of one team are marked with a CYAN box labelled A; players of the "
                    "other team with a MAGENTA box labelled B. What is the main SHIRT color of team A and of team B? "
                    "Ignore the box colors themselves. Answer ONLY with JSON: {\"A\": \"...\", \"B\": \"...\"} using one "
                    "of these French names: " + ", ".join(COULEURS_VALIDES) + ".")
        try:
            rep_g = client.models.generate_content(model=MODELE,
                                                   contents=[consigne, types.Part.from_bytes(data=jpg.tobytes(), mime_type="image/jpeg")],
                                                   config={"temperature": 0.0})
            compter(rep_g, "couleurs des équipes")
            txt = rep_g.text or ""
            d = json.loads(txt[txt.index("{"):txt.rindex("}") + 1])
            for eq_, k in ((0, "A"), (1, "B")):
                c = str(d.get(k, "")).strip().lower()
                if c in COULEURS_VALIDES:
                    votes[eq_][c] += 1
            print(f"   Gemini, coup d'envoi + {dt} s : A (équipe 0) « {d.get('A')} », B (équipe 1) « {d.get('B')} »")
        except Exception as e:
            print(f"   Gemini, coup d'envoi + {dt} s : réponse inexploitable ({e})")
    cap_g.release()
    return {eq_: v.most_common(1)[0][0] for eq_, v in votes.items() if v}


COULEUR_EQUIPE, SOURCE_COULEURS = {}, ""
if sauve and sauve.get("couleurs_source") == "gemini" and len(sauve.get("couleurs") or {}) == 2:
    COULEUR_EQUIPE = {int(k): v for k, v in sauve["couleurs"].items()}
    SOURCE_COULEURS = "gemini"
    print("   couleurs des équipes réutilisées (Gemini, sauvegarde) :", COULEUR_EQUIPE)
elif not GEMINI_KEY:
    raise SystemExit("❌ Clé Gemini absente (secret Kaggle « GEMINI_API_KEY » introuvable ou non coché pour ce notebook, "
                     "menu Add-ons > Secrets). Les couleurs des équipes viennent de Gemini ; la mesure par pixels, en "
                     "secours, n'est pas fiable (Franchimont, 29/10 : les deux équipes mesurées « rouge »).")
else:
    COULEUR_EQUIPE = couleurs_par_gemini()
    if len(COULEUR_EQUIPE) == 2:
        SOURCE_COULEURS = "gemini"
        for eq_, c in sorted(COULEUR_EQUIPE.items()):
            print(f"   équipe {eq_} : maillots « {c} » (Gemini, juste après le coup d'envoi)")

echantillons = collections.defaultdict(list)
cap = cv2.VideoCapture(VIDEO) if len(COULEUR_EQUIPE) < 2 else None
pas = max(1, len(frames) // 60)
for f in (frames[::pas] if cap is not None else []):
    # cadres ENTIÈREMENT dans l'image, et vraies détections quand time_since_update existe
    joueurs = [p for p in f.get("players", []) if p.get("team") in (0, 1) and p.get("bbox") is not None
               and not (AVEC_TSU and _tsu(p) != 0) and dans_image(p["bbox"])]
    if not joueurs:
        continue
    cap.set(cv2.CAP_PROP_POS_MSEC, (KO + f.get("frame", 0) / FPS) * 1000)
    ok, img = cap.read()
    if not ok:
        continue
    H, W = img.shape[:2]
    for p in joueurs:
        x1, y1, x2, y2 = [int(v) for v in p["bbox"]]
        h = y2 - y1
        if h < 60:
            continue
        haut = img[max(0, y1 + int(0.15 * h)):min(H, y1 + int(0.45 * h)), max(0, x1 + (x2 - x1) // 5):min(W, x2 - (x2 - x1) // 5)]
        if haut.size == 0:
            continue
        hsv = cv2.cvtColor(haut, cv2.COLOR_BGR2HSV).reshape(-1, 3).astype(int)
        # pixels de PELOUSE exclus (teintes vertes saturées) ; puis CHAQUE pixel reçoit un nom de couleur et le
        # joueur prend la couleur majoritaire
        pelouse = (hsv[:, 0] >= 30) & (hsv[:, 0] <= 90) & (hsv[:, 1] >= 40)
        maillot = hsv[~pelouse]
        if len(maillot) < 0.2 * len(hsv) or len(maillot) < 30:
            continue
        noms = collections.Counter(nom_couleur(h_, s_, v_) for h_, s_, v_ in maillot)
        echantillons[p["team"]].append(noms.most_common(1)[0][0])
if cap is not None:
    cap.release()
    print("   (mesure par pixels, en secours)")
    COULEUR_EQUIPE, SOURCE_COULEURS = {}, "pixels"
    if not echantillons:
        print("   ⚠️ Aucun échantillon de maillot exploitable (équipe en VERT, ou joueurs trop petits) : couleurs inconnues")
for eq_, lst in sorted(echantillons.items()):
    if lst:
        votes = collections.Counter(lst)
        COULEUR_EQUIPE[eq_] = votes.most_common(1)[0][0]
        print(f"   équipe {eq_} : maillots « {COULEUR_EQUIPE[eq_]} » ({len(lst)} joueurs mesurés : "
              + ", ".join(f"{c} {100 * n / len(lst):.0f} %" for c, n in votes.most_common(3)) + ")")

# ── 3. SEGMENTS CANDIDATS ─────────────────────────────────────────────────────────────────────
FAMILLES = [{"rouge", "orange"}, {"bleu", "bleu clair", "violet"}, {"blanc", "gris"}, {"jaune", "orange"}]
famille = lambda c: set().union({c}, *[fam for fam in FAMILLES if c in fam])
if EQUIPE in (0, 1):
    equipes_voulues = [EQUIPE]
else:
    proches = lambda a, b: a == b or any(a in fam and b in fam for fam in FAMILLES)
    equipes_voulues = [e for e, c in COULEUR_EQUIPE.items() if c == COULEUR.lower()] or \
                      [e for e, c in COULEUR_EQUIPE.items() if proches(c, COULEUR.lower())]
if len(equipes_voulues) != 1 or equipes_voulues[0] not in COULEUR_EQUIPE:
    raise SystemExit(f"❌ Équipe « {COULEUR} » introuvable ou ambiguë (couleurs : {COULEUR_EQUIPE}) — règle EQUIPE = 0 ou 1")
EQ = equipes_voulues[0]
if len(COULEUR_EQUIPE) < 2:
    print(f"   ⚠️ couleur de l'équipe adverse inconnue : une vignette est conforme si au moins {100 * SEUIL_COULEUR:.0f} % "
          f"du haut du maillot est « {COULEUR_EQUIPE[EQ]} » (moins discriminant)")
elif famille(COULEUR_EQUIPE[0]) & famille(COULEUR_EQUIPE[1]):
    print(f"   ⚠️ couleurs d'équipe proches ({COULEUR_EQUIPE}) : la mesure de couleur sera peu discriminante")

_FEN = [(int(h.split(":")[0]) * 60 + int(h.split(":")[1]) - FENETRE_AVANT,
         int(h.split(":")[0]) * 60 + int(h.split(":")[1]) + FENETRE_APRES) for h in FENETRES]
if BOUT_EN_BOUT and BOUT_EN_BOUT.get("bornes"):     # joueur seul : seulement le temps de jeu (KO1→fin 1re MT, KO2→fin)
    _FEN = [(float(a), float(b)) for a, b in BOUT_EN_BOUT["bornes"]]
    print(f"   temps de jeu lu : " + " ; ".join(f"{fmt(a)}-{fmt(b)}" for a, b in _FEN))
candidats = []
for sid, v in segments.items():
    if _FEN and not any(debut(sid) <= b and fin(sid) >= a for a, b in _FEN):
        continue
    if MODE == "essai" and v["tid"] not in set(map(str, TIDS_DIAG)):
        continue
    pos = v["pos"]
    nums = sorted(pos)
    lisibles = [n for n in nums if pos[n][0][3] - pos[n][0][1] >= MIN_HAUTEUR]
    if (nums[-1] - nums[0]) / FPS < MIN_DUREE_S or len(lisibles) < 2:
        continue
    part_eq = sum(1 for _, e in pos.values() if e == EQ) / len(pos)
    if MODE == "complet" and part_eq < PREFILTRE_EQUIPE:
        continue
    choix = [lisibles[int(round(q * (len(lisibles) - 1)))] for q in np.linspace(0, 1, min(N_LECTURES, len(lisibles)))]
    candidats.append((max(pos[n][0][3] - pos[n][0][1] for n in nums), sid, list(dict.fromkeys(choix))))
candidats.sort(reverse=True)
N_CANDIDATS_TOTAL = len(candidats)
if MODE == "complet" and MAX_SEGMENTS:
    candidats = candidats[:MAX_SEGMENTS]
print(f"\n   équipe {EQ} ({COULEUR_EQUIPE[EQ]}) | {len(candidats)} segment(s) à examiner sur {N_CANDIDATS_TOTAL}"
      + (f" (pistes {TIDS_DIAG} seulement, mode essai)" if MODE == "essai" else
         f" (≥ {MIN_DUREE_S} s, ≥ 2 images lisibles, ≥ {100 * PREFILTRE_EQUIPE:.0f} % de positions de l'équipe dans le cache)"))



# ── 4. COULEUR MESURÉE (méthode 2, inchangée depuis la v2.2) ──────────────────────────────────
def noms_pixels(hsv):
    """Même découpage que nom_couleur, vectorisé."""
    h, s, v = hsv[:, 0], hsv[:, 1], hsv[:, 2]
    return np.select([v < 60, (s < 45) & (v > 165), s < 45, (h < 9) | (h >= 170), h < 22, h < 34, h < 85, h < 100, h < 130],
                     ["noir", "blanc", "gris", "rouge", "orange", "jaune", "vert", "bleu clair", "bleu"], default="violet")


METHODE_COULEUR = 2      # 2 (29/10) : couleur STRICTE de l'équipe (le rouge ne compte plus l'orange, qui captait
                         # le jaune de l'arbitre) et pixels de pelouse exclus d'après la pelouse autour du joueur
MESURE_FAMILLES = [{"bleu", "bleu clair"}, {"blanc", "gris"}]
_nom_mesure_base = lambda c: set().union({c}, *[fam for fam in MESURE_FAMILLES if c in fam])
# v3.2 (07/11) : un maillot BLEU CIEL pâle est vu en grande partie « blanc / gris » par la mesure (peu saturé) ;
# 484 segments sur 661 du n°16 étaient écartés par la couleur. On compte aussi blanc et gris pour une équipe
# « bleu clair », SEULEMENT si l'autre équipe n'est ni blanche ni grise (sinon on ne pourrait plus les distinguer).
COULEUR_PALE_ELARGIE = True


def nom_mesure(c):
    fam = _nom_mesure_base(c)
    autres = [v for v in COULEUR_EQUIPE.values() if v != c]
    if COULEUR_PALE_ELARGIE and c == "bleu clair" and not any(v in ("blanc", "gris") for v in autres):
        fam = fam | {"blanc", "gris"}
    return fam


def _hsv(zone):
    return cv2.cvtColor(zone, cv2.COLOR_BGR2HSV).reshape(-1, 3).astype(int) if zone.size else np.zeros((0, 3), int)


def parts_couleur(img, b):
    """Part des pixels du haut du maillot dans la couleur de chaque équipe : [part 0, part 1].
    Pixels de pelouse retirés : référence = pixels verts juste sous les pieds et sur les côtés du cadre."""
    Hh, Ww = img.shape[:2]
    x1, y1, x2, y2 = [int(v) for v in b]
    h, w = y2 - y1, x2 - x1
    haut = img[max(0, y1 + int(0.15 * h)):max(0, y1 + int(0.45 * h)), max(0, x1 + w // 5):max(0, x2 - w // 5)]
    if haut.size == 0:
        return [0.0, 0.0]
    hsv = _hsv(haut)
    noms = noms_pixels(hsv)
    autour = np.vstack([_hsv(img[min(Hh, y2):min(Hh, y2 + max(4, h // 10)), max(0, x1):max(0, x2)]),
                        _hsv(img[max(0, y1):max(0, y2), max(0, x1 - w // 6):max(0, x1)]),
                        _hsv(img[max(0, y1):max(0, y2), min(Ww, x2):min(Ww, x2 + w // 6)])])
    garde = np.ones(len(noms), bool)
    if len(autour):
        gazon_ref = autour[noms_pixels(autour) == "vert"]
        if len(gazon_ref) > 30:
            m = np.median(gazon_ref, axis=0)
            garde = ~((np.abs(hsv[:, 0] - m[0]) <= 8) & (np.abs(hsv[:, 1] - m[1]) <= 60) & (np.abs(hsv[:, 2] - m[2]) <= 45))
    if garde.mean() < 0.25:
        return [0.0, 0.0]                         # presque que de la pelouse : cadre vide ou joueur minuscule
    noms = noms[garde]
    return [round(float(np.isin(noms, list(nom_mesure(COULEUR_EQUIPE[e]))).mean()), 3) if e in COULEUR_EQUIPE else 0.0
            for e in (0, 1)]


def equipe_mesuree(parts):
    """0, 1, ou None (arbitre, adversaire collé, ombre : indécis)."""
    a, b = parts
    if max(a, b) < SEUIL_COULEUR or max(a, b) < 1.5 * min(a, b):
        return None
    return 0 if a > b else 1


# ── 4. VIGNETTES, COULEUR MESURÉE AVANT LA QUESTION, LECTURE DU NUMÉRO ────────────────────────
import threading, bisect
_tl = threading.local()
_caps, _caps_verrou = [], threading.Lock()


def image_video(num):
    """Image suivie n° num ; une lecture de la vidéo par fil (lectures en parallèle)."""
    c = getattr(_tl, "cap", None)
    if c is None:
        c = cv2.VideoCapture(VIDEO); _tl.cap = c
        with _caps_verrou:
            _caps.append(c)
    c.set(cv2.CAP_PROP_POS_MSEC, (KO + num / FPS) * 1000)
    return c.read()


def decoupe(img, b):
    x1, y1, x2, y2 = [int(v) for v in b]; mx, my = (x2 - x1) // 6, (y2 - y1) // 10
    c = img[max(0, y1 - my):min(1080, y2 + my), max(0, x1 - mx):min(1920, x2 + mx)]
    return cv2.resize(c, (int(256 * c.shape[1] / c.shape[0]), 256)) if c.size else None


def vignettes(sid, choix):
    ims, parts, images = [], [], []
    for num in choix:
        b = segments[sid]["pos"][num][0]
        ok, img = image_video(num)
        if not ok:
            continue
        c = decoupe(img, b)
        if c is not None:
            ims.append(c); parts.append(parts_couleur(img, b)); images.append(int(num))
    return ims, parts, images


SEUIL_PARSEQ = 0.80
import sys as _sys, subprocess as _sp
_sp.run([_sys.executable, "-m", "pip", "install", "-q", "timm", "pytorch-lightning", "nltk"], check=False)
import torch as _torch
from torchvision import transforms as _T
# 06/11 : le modèle le plus récent est pris d'abord (v2 multi-matchs), sinon v1 ; code PARSeq FIGÉ (commit)
PARSEQ_COMMIT = "1902db043c029a7e03a3818c616c06600af574be"
_poids = None
for _v in ((PARSEQ_VERSION,) if PARSEQ_VERSION else ("v2", "v1")):
    _poids = next(iter(glob.glob(f"/kaggle/input/**/parseq_scoutia_maillots_{_v}.pt", recursive=True)
                       + glob.glob(f"{SORTIE}/parseq_scoutia_maillots_{_v}.pt")), None)
    if _poids:
        break
if not _poids:
    raise SystemExit("❌ parseq_scoutia_maillots_v2.pt ou _v1.pt introuvable dans les entrées")
_dev = "cuda" if _torch.cuda.is_available() else "cpu"
PARSEQ = _torch.hub.load(f"baudm/parseq:{PARSEQ_COMMIT}", "parseq", pretrained=True, trust_repo=True)
PARSEQ.load_state_dict(_torch.load(_poids, map_location="cpu")); PARSEQ = PARSEQ.eval().to(_dev)
_tfp = _T.Compose([_T.ToPILImage(), _T.Resize((32, 128), _T.InterpolationMode.BICUBIC), _T.ToTensor(), _T.Normalize(0.5, 0.5)])
_verrou_parseq = threading.Lock()
_itos = getattr(PARSEQ.tokenizer, "_itos", None)          # 06/11 : lecture restreinte aux chiffres 0-9
_MASQUE = (_torch.tensor([0.0 if (i == 0 or str(c).isdigit()) else float("-inf") for i, c in enumerate(_itos)], device=_dev)
           if _itos else None)
print(f"   lecteur de numéros : PARSeq affiné ({_poids}) sur {_dev}, seuil {SEUIL_PARSEQ}")


def _decoupes_dos(im, b):
    x1, y1, x2, y2 = [int(v) for v in b]; h, w = y2 - y1, x2 - x1
    if h < 20 or w < 8:
        return []
    H, W = im.shape[:2]; out = []
    for a, z in ((0.12, 0.55), (0.18, 0.50)):        # identiques à l'apprentissage
        c = im[max(0, y1 + int(a * h)):min(H, y1 + int(z * h)), max(0, x1 - int(0.1 * w)):min(W, x2 + int(0.1 * w))]
        if c.size:
            k = 64 / max(1, c.shape[0])
            out.append(cv2.resize(c, (max(16, int(c.shape[1] * k)), 64), interpolation=cv2.INTER_CUBIC))
    return out


def lire_parseq(dos_par_image):
    """dos_par_image : [[découpes de l'image 1], …] → [numéro ou None par image]"""
    plat = [(i, c) for i, cs in enumerate(dos_par_image) for c in cs]
    res = [None] * len(dos_par_image)
    if not plat:
        return res
    with _verrou_parseq, _torch.no_grad():
        x = _torch.stack([_tfp(cv2.cvtColor(c, cv2.COLOR_BGR2RGB)) for _, c in plat]).to(_dev)
        _lg = PARSEQ(x)
        if _MASQUE is not None and _lg.shape[-1] == _MASQUE.shape[0]:
            _lg = _lg + _MASQUE
        lab, conf = PARSEQ.tokenizer.decode(_lg.softmax(-1))
    meilleur = {}
    for (i, _), l, cf in zip(plat, lab, conf):
        cf = float(cf.prod()); d = "".join(ch for ch in l if ch.isdigit())
        if cf >= SEUIL_PARSEQ and 1 <= len(d) <= 2 and (i not in meilleur or cf > meilleur[i][1]):
            meilleur[i] = (int(d), cf)
    for i, (n, _) in meilleur.items():
        res[i] = n
    return res


compteur = {"questions": 0, "evitees": 0, "plafond": 0}
verrou_compteur = threading.Lock()


def lire(sid, choix):
    ims, parts, images = vignettes(sid, choix)
    entree = {"choix": [int(n) for n in choix], "images": images, "parts": parts, "numeros": None,
              "couleur_v": METHODE_COULEUR, "source": f"lectures_v30_{MATCH}.json",
              "reflexion": REFLEXION_LECTURE or "normale"}
    if sum(1 for p in parts if equipe_mesuree(p) == EQ) < 1:          # 07/11 : lu dès 1 vignette de la bonne couleur (gratuit) ; règle de retenue inchangée
        entree["numeros"], entree["non_lu"] = [None] * len(ims), "couleur"     # ne peut pas être retenu : pas de question
        with verrou_compteur:
            compteur["evitees"] += 1
        return sid, entree, ims
    with verrou_compteur:
        if compteur["questions"] >= MAX_QUESTIONS:
            compteur["plafond"] += 1
            entree["numeros"], entree["non_lu"] = [None] * len(ims), "plafond"   # couleur gardée, lu au run suivant
            return sid, entree, ims
        compteur["questions"] += 1
    dos = []
    for num in images:                                           # v3.0 : lecture par PARSeq affiné, sur le dos
        ok, img = image_video(num)
        dos.append(_decoupes_dos(img, segments[sid]["pos"][num][0]) if ok else [])
    try:
        entree["numeros"] = lire_parseq(dos)
        entree["source"] = f"lectures_v30_{MATCH}.json (PARSeq affiné)"
        return sid, entree, ims
    except Exception as e:
        print(f"   ⚠️ lecture PARSeq {sid} : {e}")
        return sid, None, ims


def a_relire(sid, choix):
    e = lectures.get(sid)
    if not e or e.get("choix") != choix:
        return True
    if e.get("non_lu") == "plafond":
        return True                                                         # question pas encore posée
    return bool(e.get("non_lu")) and e.get("couleur_v") != METHODE_COULEUR   # couleur à remesurer avec la méthode actuelle


a_lire = [c for c in candidats if a_relire(c[1], c[2])]
_a_lire = {c[1] for c in a_lire}
N_REUTILISEES_ICI = sum(1 for _, sid, _ in candidats if sid not in _a_lire and _lue(lectures.get(sid) or {}))
N_EVITEES_CONNUES = sum(1 for _, sid, _ in candidats if sid not in _a_lire and (lectures.get(sid) or {}).get("non_lu") == "couleur")
VIGN, erreurs = {}, set()
print(f"   lectures : {len(candidats) - len(a_lire)} segment(s) déjà connu(s) | {len(a_lire)} à examiner "
      f"(couleur d'abord ; question seulement si ≥ 2 vignettes « {COULEUR_EQUIPE[EQ]} », au plus {MAX_QUESTIONS})")
# ── 09/11 : LECTURE RAPIDE ─────────────────────────────────────────────────────────────────────────────────────
# La vidéo est lue UNE fois, dans l'ordre (plus de saut d'une image à l'autre), en 2 processus (une moitié du temps
# chacun) qui découpent les vignettes, mesurent la couleur et préparent les découpes du dos ; PARSeq lit ensuite tout
# sur le GPU par lots. TOUS les morceaux candidats sont lus (les deux équipes) : le fichier de lectures sert ensuite
# au montage de N'IMPORTE QUEL joueur du match sans relire la vidéo. Retenue d'un morceau : règle inchangée.
LECTURE_RAPIDE = True
N_PROCESSUS_LECTURE = 2


def _lire_tranche(taches):
    """taches : [(n° d'image du suivi, [(sid, idx, cadre), …]), …] triées → [(sid, idx, n°, parts, [dos jpg])]"""
    cv2.setNumThreads(1)                            # processus fils : pas de sur-parallélisme
    cap = cv2.VideoCapture(VIDEO)
    fps_v = cap.get(cv2.CAP_PROP_FPS) or FPS
    out, cur = [], None
    for num, items in taches:
        cible = int(round((KO + num / FPS) * fps_v))
        if cur is None or cible < cur or cible - cur > 300:
            cap.set(cv2.CAP_PROP_POS_FRAMES, cible); cur = cible
        while cur < cible:
            cap.grab(); cur += 1
        ok, img = cap.read(); cur += 1
        if not ok:
            continue
        for sid, idx, b in items:
            dos = [cv2.imencode(".png", d)[1].tobytes() for d in _decoupes_dos(img, b)]
            out.append((sid, idx, int(num), parts_couleur(img, b), dos))
    cap.release()
    return out


if a_lire and LECTURE_RAPIDE:
    import multiprocessing as _mp
    t_l = time.time()
    _besoin = collections.defaultdict(list)
    for _, sid, choix in a_lire:
        for idx, num in enumerate(choix):
            _besoin[int(num)].append((sid, idx, segments[sid]["pos"][num][0]))
    _taches = sorted(_besoin.items())
    _n = max(1, N_PROCESSUS_LECTURE)
    _morceaux = [_taches[i * len(_taches) // _n:(i + 1) * len(_taches) // _n] for i in range(_n)]
    with _mp.get_context("fork").Pool(_n) as _pool:
        _res = [r for part in _pool.map(_lire_tranche, _morceaux) for r in part]
    t_dec = time.time() - t_l
    _par_seg = collections.defaultdict(dict)
    for sid, idx, num, parts, dos in _res:
        _par_seg[sid][idx] = (num, parts, [cv2.imdecode(np.frombuffer(d, np.uint8), cv2.IMREAD_COLOR) for d in dos])
    # PARSeq par lots (toutes les images de tous les morceaux)
    _ordre = [(sid, idx) for sid in _par_seg for idx in sorted(_par_seg[sid])]
    _lus = {}
    for k in range(0, len(_ordre), 400):
        lot = _ordre[k:k + 400]
        nums = lire_parseq([_par_seg[sid][idx][2] for sid, idx in lot])
        for (sid, idx), n in zip(lot, nums):
            _lus[(sid, idx)] = n
    for _, sid, choix in a_lire:
        d = _par_seg.get(sid, {})
        idxs = sorted(d)
        lectures[sid] = {"choix": [int(n) for n in choix], "images": [d[i][0] for i in idxs], "parts": [d[i][1] for i in idxs],
                         "numeros": [_lus.get((sid, i)) for i in idxs], "couleur_v": METHODE_COULEUR,
                         "source": f"lectures_v30_{MATCH}.json (PARSeq affiné, lecture rapide)", "reflexion": "parseq"}
    compteur["questions"] = len(a_lire)
    print(f"   LECTURE RAPIDE : {len(a_lire)} morceau(x), {len(_res)} images lues en {time.time() - t_l:.0f} s "
          f"(vidéo {t_dec:.0f} s en {_n} processus, PARSeq {time.time() - t_l - t_dec:.0f} s) — les deux équipes, réutilisable "
          f"pour tout joueur du match")
elif a_lire:
    import concurrent.futures
    client = types = None
    # v3.0 : pas de Gemini pour la lecture des numéros (PARSeq affiné)
    t_l = time.time()
    with concurrent.futures.ThreadPoolExecutor(4) as ex:
        for sid, entree, ims in ex.map(lambda c: lire(c[1], c[2]), a_lire):
            VIGN[sid] = ims
            if entree is None:
                erreurs.add(sid)
            else:
                lectures[sid] = entree
    print(f"   {len(a_lire)} segment(s) examiné(s) en {time.time() - t_l:.0f} s | questions posées : {compteur['questions']} | "
          f"évitées par la couleur : {compteur['evitees']} | non posées (plafond) : {compteur['plafond']} | erreurs : {len(erreurs)}")
# couleurs mesurées avec une ancienne méthode : remesurées sur la vidéo (aucune question Gemini)
a_remesurer = [sid for _, sid, _ in candidats if sid in lectures and sid not in VIGN
               and lectures[sid].get("couleur_v") != METHODE_COULEUR]
for sid in a_remesurer:
    ims, parts, images = vignettes(sid, lectures[sid]["images"])
    if images == lectures[sid]["images"]:
        lectures[sid]["parts"], lectures[sid]["couleur_v"] = parts, METHODE_COULEUR
        VIGN[sid] = ims
if a_remesurer:
    print(f"   couleurs remesurées (nouvelle méthode, sans Gemini) : {len(a_remesurer)} segment(s)")
sauver_lectures()
print(f"   lectures sauvegardées : {FICHIER_LECTURES} (à garder : sortie du notebook, puis dataset en entrée)")


def evaluer(sid):
    """Portions retenues d'un segment : [(image de début, image de fin)], et la raison du verdict."""
    e = lectures.get(sid)
    if not e or e.get("numeros") is None:
        return [], "plafond de questions atteint" if sid not in erreurs and e is None else "question en erreur"
    if e.get("non_lu") == "plafond":
        return [], "plafond de questions atteint (à lire au prochain run)"
    if e.get("non_lu"):
        return [], "moins de 2 vignettes de la bonne couleur (question évitée)"
    vig = sorted(zip(e["images"], e["numeros"], [equipe_mesuree(p) for p in e["parts"]]))
    if not vig:
        return [], "aucune vignette"
    pos = segments[sid]["pos"]
    portions, suites, courant = [], [], []
    for i, v in enumerate(vig):
        # vignette conforme : bonne couleur ET pas d'autre numéro lu (v2.5 : un autre numéro, même lu une seule
        # fois, arrête la portion comme une vignette de mauvaise couleur)
        if v[2] == EQ and (v[1] is None or v[1] == NUMERO):
            courant.append(i)
        elif courant:
            suites.append(courant); courant = []
    if courant:
        suites.append(courant)
    for s in suites:
        c = collections.Counter(vig[i][1] for i in s if vig[i][1] is not None)
        autre = any(n != NUMERO and k >= MIN_CONCORDANTES for n, k in c.items())
        ok = c[NUMERO] >= MIN_CONCORDANTES and not autre
        if not ok and LECTURE_UNIQUE_SI_COULEUR:
            ok = c[NUMERO] >= 1 and len(c) == 1 and len(s) >= 2
        if ok:
            # v2.6 : prolongée au plus de EXTENSION_MAX_S au-delà des vignettes lues, et seulement si aucune vignette
            # non conforme n'est au-delà ; jamais au-delà des bords du segment
            ext = EXTENSION_MAX_S * FPS
            a = max(min(pos), vig[s[0]][0] - ext) if s[0] == 0 else vig[s[0]][0]
            b = min(max(pos), vig[s[-1]][0] + ext) if s[-1] == len(vig) - 1 else vig[s[-1]][0]
            portions.append((a, b))
    lus = [v[1] for v in vig if v[1] is not None]
    if portions:
        coupe = any(v[1] is not None and v[1] != NUMERO for v in vig)
        return portions, "retenu" + (" (coupé à un autre numéro)" if coupe else "")
    if NUMERO not in lus:
        return [], "autre numéro" if lus else "aucun numéro lisible"
    if not any(v[2] == EQ for v in vig):
        return [], f"{NUMERO} lu, mais aucune vignette de la bonne couleur"
    return [], f"{NUMERO} lu moins de {MIN_CONCORDANTES} fois sur des vignettes consécutives de la bonne couleur sans autre numéro"


verdicts = {sid: evaluer(sid) for _, sid, _ in candidats}

# ── v3.2 : CONFIRMATION GEMINI des segments où le numéro n'a été lu qu'une fois ──
CONFIRMATION_GEMINI, MAX_CONFIRMATIONS = True, 80
_a_conf = [sid for sid, (p, r) in verdicts.items() if not p and str(r).startswith(f"{NUMERO} lu moins")]
if CONFIRMATION_GEMINI and _a_conf and GEMINI_KEY:
    from google import genai as _gg
    from google.genai import types as _gt
    _cl = _gg.Client(api_key=GEMINI_KEY); _jet = {"q": 0, "e": 0, "s": 0}; _gagnes = 0
    _Q = """These images show (normally) the same amateur football player at several moments. For EACH image, give the
shirt number ONLY if its digits are clearly readable (back, front or shorts); otherwise null. Do not guess.
Answer ONLY with JSON: {"numbers": [one value per image, in order]}"""
    for sid in _a_conf[:MAX_CONFIRMATIONS]:   # (12 images par morceau : la planche envoyée à Gemini est plus large)
        e = lectures[sid]
        try:
            ims, _, _ = vignettes(sid, e["choix"])
            h = 200; pl = cv2.hconcat([cv2.resize(i, (max(8, int(i.shape[1] * h / i.shape[0])), h)) for i in ims])
            r = _cl.models.generate_content(model=MODELE, contents=[_Q, _gt.Part.from_bytes(
                data=cv2.imencode(".jpg", pl, [cv2.IMWRITE_JPEG_QUALITY, 90])[1].tobytes(), mime_type="image/jpeg")],
                config={"temperature": 0.0, "thinking_config": {"thinking_level": "low"}})
            um = getattr(r, "usage_metadata", None); _jet["q"] += 1
            _jet["e"] += int(getattr(um, "prompt_token_count", 0) or 0)
            _jet["s"] += int(getattr(um, "candidates_token_count", 0) or 0) + int(getattr(um, "thoughts_token_count", 0) or 0)
            txt = r.text or ""; d, _ = json.JSONDecoder().raw_decode(txt[txt.index("{"):])
            gem = [int(x) if str(x).isdigit() else None for x in (d.get("numbers") or [])][:len(e["numeros"])]
            gem += [None] * (len(e["numeros"]) - len(gem))
            e["numeros_parseq"] = list(e["numeros"])
            e["numeros"] = [g if g is not None else p for g, p in zip(gem, e["numeros"])]   # Gemini prime s'il lit
            e["source"] = str(e.get("source")) + " + confirmation Gemini"
            verdicts[sid] = evaluer(sid); _gagnes += bool(verdicts[sid][0])
        except Exception as ex:
            print(f"   ⚠️ confirmation {sid} : {ex}")
    print(f"   CONFIRMATION GEMINI : {len(_a_conf)} segment(s) lus une seule fois | {_jet['q']} relu(s) | "
          f"{_gagnes} désormais retenu(s) | ≈ {_jet['e'] / 1e6 * 0.75 + _jet['s'] / 1e6 * 3.75:.3f} $")
# ── v3.2 : DIAGNOSTIC D'UNE FENÊTRE (gratuit) : où la chaîne casse-t-elle pour ce joueur ? ──
DIAG_FENETRES = [] if BOUT_EN_BOUT else [("81:50", "82:02"), ("87:15", "87:30")]   # [] = pas de diagnostic (n°16 vu à 81:56 et 87:21)
for DIAG_FENETRE in DIAG_FENETRES:
    _s = lambda h: int(h.split(":")[0]) * 60 + int(h.split(":")[1])
    _a, _b = _s(DIAG_FENETRE[0]), _s(DIAG_FENETRE[1])
    _cand = {sid: ch for _, sid, ch in candidats}
    print(f"\n   DIAGNOSTIC {DIAG_FENETRE[0]}-{DIAG_FENETRE[1]} : morceaux du suivi qui touchent la fenêtre")
    _lignes = []
    for sid, v in sorted(segments.items(), key=lambda x: debut(x[0])):
        if not (debut(sid) <= _b and fin(sid) >= _a):
            continue
        pos = v["pos"]; nums = sorted(pos)
        dur = (nums[-1] - nums[0]) / FPS
        h_med = float(np.median([pos[n][0][3] - pos[n][0][1] for n in nums]))
        part_eq = sum(1 for _, e in pos.values() if e == EQ) / len(pos)
        if sid in _cand:
            e = lectures.get(sid) or {}
            raison = verdicts.get(sid, ([], "?"))[1]
            detail = f"lu : {e.get('numeros')} | couleurs : {[equipe_mesuree(p) for p in e.get('parts') or []]} | {raison}"
            choix = _cand[sid]
        else:
            lis = [n for n in nums if pos[n][0][3] - pos[n][0][1] >= MIN_HAUTEUR]
            why = ("trop court (< %.1f s)" % MIN_DUREE_S if dur < MIN_DUREE_S else
                   "moins de 2 images assez grandes" if len(lis) < 2 else
                   f"part de l'équipe {part_eq:.0%} < {PREFILTRE_EQUIPE:.0%}" if part_eq < PREFILTRE_EQUIPE else "non retenu (plafond ?)")
            detail = f"NON EXAMINÉ : {why}"
            choix = [nums[int(round(q * (len(nums) - 1)))] for q in np.linspace(0, 1, min(4, len(nums)))]
        print(f"     {sid:24s} {fmt(debut(sid))}-{fmt(fin(sid))} ({dur:4.1f} s, hauteur ~{h_med:.0f} px, équipe {part_eq:.0%}) | {detail}")
        try:
            ims, _, _ = vignettes(sid, choix)
            if ims:
                hh = 220; l = cv2.hconcat([cv2.resize(i, (max(8, int(i.shape[1] * hh / i.shape[0])), hh)) for i in ims])
                cv2.putText(l, f"{fmt(debut(sid))} {sid[:14]}", (3, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
                _lignes.append(l)
        except Exception:
            pass
    if _lignes:                                    # 07/11 : pages de 12 lignes (une seule image trop haute était réduite à l'envoi)
        _w = max(l.shape[1] for l in _lignes)
        for _k in range(0, len(_lignes), 12):
            _nomf = f"{SORTIE}/diagnostic_fenetre_{NUMERO}_{DIAG_FENETRE[0].replace(':', '-')}_p{_k // 12 + 1}.jpg"
            cv2.imwrite(_nomf, np.vstack([np.pad(l, ((0, 4), (0, _w - l.shape[1]), (0, 0))) for l in _lignes[_k:_k + 12]]))
        print(f"     planches : diagnostic_fenetre_{NUMERO}_{DIAG_FENETRE[0].replace(':', '-')}_p1…p{(len(_lignes) - 1) // 12 + 1}.jpg "
              f"(12 morceaux par page, dans l'ordre ci-dessus ; lignes 1-12 en p1, 13-24 en p2…)")

retenus = [sid for sid, (p, _) in verdicts.items() if p]
rejets = collections.Counter(r for p, r in verdicts.values() if not p)
print(f"   RETENUS (numéro lu) pour le {NUMERO} {COULEUR} : {len(retenus)} segment(s) | écartés : {dict(rejets)}")

# planche des lectures
nom_eq = lambda q: COULEUR_EQUIPE.get(q, "?") if q is not None else "?"
douteux = [sid for _, sid, _ in candidats if sid not in retenus and NUMERO in ((lectures.get(sid) or {}).get("numeros") or [])]
lignes_v = []


import unicodedata
ascii_ = lambda t: unicodedata.normalize("NFKD", str(t)).encode("ascii", "ignore").decode()   # OpenCV : pas d'accents


def ligne_planche(titre, coul, cases):
    titre = ascii_(titre)
    l = cv2.hconcat(cases)
    l = cv2.copyMakeBorder(l, 40, 0, 0, max(0, 760 - l.shape[1]), cv2.BORDER_CONSTANT, value=(30, 30, 30))
    cv2.putText(l, titre[:95], (6, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.5, coul, 1)
    return l[:, :760]


def case(im, texte, bon):
    c = cv2.copyMakeBorder(cv2.resize(im, (120, 256)), 0, 26, 0, 0, cv2.BORDER_CONSTANT, value=(30, 30, 30))
    cv2.putText(c, ascii_(texte), (4, 276), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0) if bon else (0, 0, 255), 1)
    return c


def ecrire_planches(lignes, base):
    """Pages de LIGNES_PAR_PAGE lignes : base_p1.jpg, base_p2.jpg… Renvoie la liste des fichiers écrits."""
    for ancien in glob.glob(f"{base}_p*.jpg"):
        os.remove(ancien)
    pages = [lignes[k:k + LIGNES_PAR_PAGE] for k in range(0, len(lignes), LIGNES_PAR_PAGE)]
    noms = []
    for i, p in enumerate(pages, 1):
        noms.append(f"{base}_p{i}.jpg"); cv2.imwrite(noms[-1], cv2.vconcat(p))
    return noms


for sid, etiquette, coul in [(s, "RETENU", (0, 200, 0)) for s in sorted(retenus, key=debut)] + \
                           [(s, "ECARTE", (0, 0, 255)) for s in sorted(douteux, key=debut)]:
    e = lectures[sid]
    ims = VIGN.get(sid) or vignettes(sid, e["images"])[0]
    if not ims:
        continue
    cases = [case(im, f"{n if n is not None else '-'} {nom_eq(equipe_mesuree(p))}", equipe_mesuree(p) == EQ)
             for im, n, p in zip(ims, e["numeros"], e["parts"])]
    portions = verdicts[sid][0]
    txt = f"{etiquette} | {sid} | {fmt(debut(sid))}-{fmt(fin(sid))} | " + (
        "garde " + ", ".join(f"{fmt(KO + a / FPS)}-{fmt(KO + b / FPS)}" for a, b in portions) if portions else verdicts[sid][1])
    lignes_v.append(ligne_planche(txt, coul, cases))
if lignes_v:
    pages = ecrire_planches(lignes_v, f"{SORTIE}/verification_lectures_v30")
    print(f"   planche des lectures : {len(pages)} page(s), {len(retenus)} retenus puis {len(douteux)} écartés où le "
          f"{NUMERO} est lu ({os.path.basename(pages[0])}…)")
with open(f"{SORTIE}/segments_v28_{MATCH}_{NUMERO}_{COULEUR}.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f, delimiter=";")
    w.writerow(["segment", "piste", "debut", "fin", "coupe_avant", "images_lues", "numeros", "couleurs_mesurees", "verdict", "portions"])
    for _, sid, _ in sorted(candidats, key=lambda c: debut(c[1])):
        e = lectures.get(sid) or {}
        w.writerow([sid, segments[sid]["tid"], fmt(debut(sid)), fmt(fin(sid)), segments[sid]["raison"] or "",
                    " ".join(fmt(KO + n / FPS) for n in e.get("images", [])),
                    " ".join("-" if n is None else str(n) for n in (e.get("numeros") or [])),
                    " ".join(nom_eq(equipe_mesuree(p)) for p in e.get("parts", [])),
                    verdicts[sid][1], " ".join(f"{fmt(KO + a / FPS)}-{fmt(KO + b / FPS)}" for a, b in verdicts[sid][0])])
if not retenus:
    raise SystemExit(f"❌ Aucun segment où le {NUMERO} est lu de façon concordante — rien à monter (aucune valeur devinée)")


# ── 4 bis. RACCORDEMENT DES SEGMENTS VOISINS (sans Gemini) ────────────────────────────────────
nums_seg = {sid: sorted(v["pos"]) for sid, v in segments.items()}
par_debut = sorted((n[0], sid) for sid, n in nums_seg.items())
par_fin = sorted((n[-1], sid) for sid, n in nums_seg.items())
_cles_debut = [d for d, _ in par_debut]
_cles_fin = [f for f, _ in par_fin]


def _bas(b):
    return (b[0] + b[2]) / 2, b[3], max(1.0, b[3] - b[1])


def distance_h(b1, b2):
    x1, y1, h1 = _bas(b1); x2, y2, h2 = _bas(b2)
    return float(np.hypot(x1 - x2, y1 - y2) / ((h1 + h2) / 2))


def couleurs_segment(sid):
    """[(image, équipe mesurée)] dans l'ordre du temps : lectures existantes, sinon 3 vignettes (début, milieu, fin)."""
    e = lectures.get(sid)
    if e and e.get("couleur_v") == METHODE_COULEUR and e.get("parts"):
        return sorted((n, equipe_mesuree(p)) for n, p in zip(e["images"], e["parts"]))
    if sid in COUL_R and COUL_R[sid].get("couleur_v") == METHODE_COULEUR:
        return sorted((n, equipe_mesuree(p)) for n, p in zip(COUL_R[sid]["images"], COUL_R[sid]["parts"]))
    pos, nums = segments[sid]["pos"], nums_seg[sid]
    grands = [n for n in nums if pos[n][0][3] - pos[n][0][1] >= RACCORD_HMIN] or nums
    choix = list(dict.fromkeys([grands[0], grands[len(grands) // 2], grands[-1]]))
    images, parts = [], []
    for n in choix:
        ok, img = image_video(n)
        if ok:
            images.append(int(n)); parts.append(parts_couleur(img, pos[n][0]))
    COUL_R[sid] = {"images": images, "parts": parts, "couleur_v": METHODE_COULEUR}
    return sorted((n, equipe_mesuree(p)) for n, p in zip(images, parts))


def contredit(sid):
    """Un autre numéro a-t-il été lu sur une vignette de la bonne couleur de ce segment ?"""
    e = lectures.get(sid) or {}
    return any(n is not None and n != NUMERO and equipe_mesuree(p) == EQ
               for n, p in zip(e.get("numeros") or [], e.get("parts") or []))


def voisins(sid_a, bord, sens):
    """Segments plausibles au raccord : (segment, écart en s, distance en hauteurs, image du raccord)."""
    bb_a = segments[sid_a]["pos"][bord][0]
    out = []
    if sens > 0:
        i0 = bisect.bisect_left(_cles_debut, bord - RACCORD_RECOUV_S * FPS)
        i1 = bisect.bisect_right(_cles_debut, bord + RACCORD_MAX_S * FPS)
        cands = [sid for _, sid in par_debut[i0:i1] if sid != sid_a and nums_seg[sid][-1] > bord]
    else:
        i0 = bisect.bisect_left(_cles_fin, bord - RACCORD_MAX_S * FPS)
        i1 = bisect.bisect_right(_cles_fin, bord + RACCORD_RECOUV_S * FPS)
        cands = [sid for _, sid in par_fin[i0:i1] if sid != sid_a and nums_seg[sid][0] < bord]
    pos_a = segments[sid_a]["pos"]
    for sid in cands:
        nums, pos = nums_seg[sid], segments[sid]["pos"]
        j = nums[0] if sens > 0 else nums[-1]
        chevauche = (sens > 0 and j <= bord) or (sens < 0 and j >= bord)
        if chevauche:                                                   # chevauchement : comparer à la même image
            communs = [n for n in nums if n in pos_a]
            if communs:
                n = min(communs, key=lambda k: abs(k - bord))
                d, ecart = distance_h(pos_a[n][0], pos[n][0]), 0.0
            else:
                n = min(nums, key=lambda k: abs(k - bord))
                d, ecart = distance_h(bb_a, pos[n][0]), 0.0
        else:
            d, ecart = distance_h(bb_a, pos[j][0]), abs(j - bord) / FPS
        if d <= RACCORD_BASE_H + RACCORD_H_PAR_S * ecart:
            out.append((sid, round(ecart, 2), round(d, 2), j, chevauche))
    return out


def statut_couleur(cs):
    n_eq = sum(1 for _, q in cs if q == EQ)
    n_autre = sum(1 for _, q in cs if q is not None and q != EQ)
    if n_autre > n_eq:
        return "autre équipe"
    return "bonne couleur" if n_eq >= 2 and n_autre == 0 else "incertaine"


VARIANTES = {"ref": (False, True), "B": (True, True), "B partiel (v2.4)": (True, False)}   # (plus proche, entier seul)
ABREGE = {"ref": "R", "B": "B", "B partiel (v2.4)": "Bp"}
if RACCORD_MONTAGE not in ("ref", "B"):
    raise SystemExit(f"❌ RACCORD_MONTAGE = {RACCORD_MONTAGE!r} : choisis « ref » ou « B »")
_infos_cache = {}


def infos_voisins(sid_a, bord, sens):
    """Voisins plausibles avec leur couleur : (segment, écart s, distance h, couleurs, statut, nb positions).
    Mis en cache : les quatre variantes partagent exactement les mêmes mesures."""
    cle = (sid_a, bord, sens)
    if cle not in _infos_cache:
        brut = voisins(sid_a, bord, sens)
        # v2.6 : deux pistes présentes au même moment et éloignées = deux joueurs différents (pas de couleur mesurée)
        _infos_cache[cle] = [(sid, ecart, d, [] if (chev and d > RACCORD_RECOUV_MAX_H) else couleurs_segment(sid),
                              "chevauchement éloigné" if (chev and d > RACCORD_RECOUV_MAX_H) else None,
                              len(nums_seg[sid])) for sid, ecart, d, j, chev in brut]
        _infos_cache[cle] = [(i[0], i[1], i[2], i[3], i[4] or statut_couleur(i[3]), i[5]) for i in _infos_cache[cle]]
    return _infos_cache[cle]


def raccorder(nom_variante):
    """Raccords d'une variante. Renvoie (raccords [(segment, début, fin)], trace [une ligne par décision])."""
    plus_proche, entier_seul = VARIANTES[nom_variante]
    raccords, trace = [], []
    couverts = {sid: list(p) for sid, (p, _) in verdicts.items() if p}

    def chercher(sid_a, bord, sens, profondeur, origine):
        if profondeur > RACCORD_MAX_CHAINE:
            return
        infos = infos_voisins(sid_a, bord, sens)
        concurrents = [i for i in infos if i[4] not in ("autre équipe", "chevauchement éloigné")]
        ligne = {"variante": nom_variante, "origine": origine, "depuis": sid_a, "sens": "après" if sens > 0 else "avant",
                 "profondeur": profondeur, "bord": fmt(KO + bord / FPS),
                 "voisins": " ".join(f"{i[0]}({i[4]},{i[5]}pos,{i[2]}h,{i[1]}s)" for i in infos)}
        if not concurrents:
            trace.append({**ligne, "decision": "aucun voisin plausible"}); return
        regle = ""
        if len(concurrents) > 1:
            tri = sorted(concurrents, key=lambda i: i[2])
            if plus_proche and tri[1][2] >= RACCORD_RAPPORT * max(tri[0][2], 0.05):   # B
                concurrents, regle = [tri[0]], f" [B : {tri[0][2]} h contre {tri[1][2]} h]"
            else:
                trace.append({**ligne, "decision": f"ambigu ({len(concurrents)} candidats non exclus par la couleur)"}); return
        sid, ecart, d, cs, statut, _ = concurrents[0]
        ligne.update({"candidat": sid, "ecart_s": ecart, "distance_h": d, "couleurs": " ".join(nom_eq(q) for _, q in cs)})
        if sid in couverts:
            trace.append({**ligne, "decision": "déjà retenu (lien seulement)" + regle}); return
        if statut != "bonne couleur":
            trace.append({**ligne, "decision": f"refusé : couleur {statut}" + regle}); return
        if contredit(sid):
            trace.append({**ligne, "decision": "refusé : autre numéro lu sur ce segment" + regle}); return
        nums = nums_seg[sid]
        ordre = cs if sens > 0 else cs[::-1]                 # du côté du raccord vers l'autre bout
        bons = 0
        while bons < len(ordre) and ordre[bons][1] == EQ:
            bons += 1
        if bons == 0:
            trace.append({**ligne, "decision": "refusé : mauvaise couleur au raccord" + regle}); return
        entier = bons == len(ordre)
        if entier_seul and not entier:
            trace.append({**ligne, "decision": "refusé : couleur incomplète (raccord en partie)" + regle}); return
        if sens > 0:
            a, b = nums[0], (nums[-1] if entier else ordre[bons - 1][0])
        else:
            a, b = (nums[0] if entier else ordre[bons - 1][0]), nums[-1]
        raccords.append((sid, a, b)); couverts[sid] = [(a, b)]
        trace.append({**ligne, "decision": "ACCEPTÉ" + ("" if entier else " (en partie)") + regle,
                      "portion": f"{fmt(KO + a / FPS)}-{fmt(KO + b / FPS)}"})
        if entier:
            chercher(sid, nums[-1] if sens > 0 else nums[0], sens, profondeur + 1, origine)

    for sid in sorted(retenus, key=debut):
        for a, b in verdicts[sid][0]:
            nums = nums_seg[sid]
            if b == nums[-1]:          # la portion va jusqu'au bout du segment : on peut chercher la suite
                chercher(sid, b, +1, 1, sid)
            if a == nums[0]:
                chercher(sid, a, -1, 1, sid)
    return raccords, trace


t_r = time.time()
RESULTATS = {v: raccorder(v) for v in VARIANTES}
sauver_lectures()                  # couleurs mesurées pour les raccords conservées pour le prochain run
cat = lambda d: "accepté" if d.startswith("ACCEPTÉ") else "ambigu" if d.startswith("ambigu") else \
    "aucun voisin" if d.startswith("aucun") else "déjà retenu" if d.startswith("déjà") else "refusé"
print(f"\n   RACCORDS (sans Gemini, {time.time() - t_r:.0f} s) — comparaison des variantes sur les mêmes lectures :")
print(f"   {'variante':<8} {'acceptés':>9} {'ambigus':>8} {'refusés':>8} {'aucun':>6} {'déjà ret.':>9} {'durée ajoutée':>14}")
COMPARAISON = []
for v, (rac, tr) in RESULTATS.items():
    c = collections.Counter(cat(t["decision"]) for t in tr)
    duree_aj = sum((b - a) / FPS for _, a, b in rac)
    COMPARAISON.append([v, len(rac), c["ambigu"], c["refusé"], c["aucun voisin"], c["déjà retenu"], round(duree_aj, 1)])
    print(f"   {v:<8} {len(rac):>9} {c['ambigu']:>8} {c['refusé']:>8} {c['aucun voisin']:>6} {c['déjà retenu']:>9} {duree_aj:>12.0f} s"
          + ("   ← montage" if v == RACCORD_MONTAGE else ""))
with open(f"{SORTIE}/raccords_v28_{MATCH}_{NUMERO}_{COULEUR}.csv", "w", newline="", encoding="utf-8") as f:
    champs = ["variante", "origine", "depuis", "sens", "profondeur", "bord", "candidat", "ecart_s", "distance_h", "couleurs",
              "decision", "portion", "voisins"]
    w = csv.DictWriter(f, fieldnames=champs, delimiter=";", extrasaction="ignore")
    w.writeheader()
    for v, (_, tr) in RESULTATS.items():
        w.writerows(tr)
with open(f"{SORTIE}/raccords_comparaison_v28_{MATCH}_{NUMERO}_{COULEUR}.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f, delimiter=";")
    w.writerow(["variante", "acceptes", "ambigus", "refuses", "aucun_voisin", "deja_retenus", "duree_ajoutee_s"])
    w.writerows(COMPARAISON)
# segment raccordé -> variantes qui l'acceptent (affiché sur la planche et dans le montage)
VAR_PAR_SEG = collections.defaultdict(list)
for v, (rac, _) in RESULTATS.items():
    for sid, _, _ in rac:
        VAR_PAR_SEG[sid].append(v)
abrege = lambda vs: ",".join(ABREGE[v] for v in vs)
# planche : chaque raccord accepté par au moins une variante, une seule fois, avec les variantes qui l'acceptent
lignes_r, vus = [], set()
for v in VARIANTES:
    for t in [t for t in RESULTATS[v][1] if t["decision"].startswith("ACCEPTÉ")]:
        if t["candidat"] in vus:
            continue
        vus.add(t["candidat"])
        sid_a, sid = t["depuis"], t["candidat"]
        n_a = nums_seg[sid_a][-1] if t["sens"] == "après" else nums_seg[sid_a][0]
        cases = []
        ok, img = image_video(n_a)
        if ok and decoupe(img, segments[sid_a]["pos"][n_a][0]) is not None:
            cases.append(case(decoupe(img, segments[sid_a]["pos"][n_a][0]), "origine", True))
        for n, q in couleurs_segment(sid):
            ok, img = image_video(n)
            if ok and decoupe(img, segments[sid]["pos"][n][0]) is not None:
                cases.append(case(decoupe(img, segments[sid]["pos"][n][0]), f"{fmt(KO + n / FPS)} {nom_eq(q)}", q == EQ))
        if cases:
            lignes_r.append(ligne_planche(f"[{abrege(VAR_PAR_SEG[sid])}] {t['sens']} | {sid_a} -> {sid} | {t['ecart_s']} s, "
                                          f"{t['distance_h']} h | {t.get('portion', '')}", (255, 200, 0), cases))
if lignes_r:
    pages = ecrire_planches(lignes_r, f"{SORTIE}/verification_raccords_v28")
    print(f"   planche des raccords : {len(pages)} page(s) ([variantes qui l'acceptent : R = ref, B, Bp = B partiel v2.4], "
          f"1re vignette = origine)")
raccords = RESULTATS[RACCORD_MONTAGE][0]

# ── 5. MONTAGE AVEC REPÈRE (portions lues en jaune, raccords en bleu avec leurs variantes) ──────
positions, spans = {}, []
for sid in retenus:
    for a, b in verdicts[sid][0]:
        spans.append((KO + a / FPS, KO + b / FPS))
        for num, (bb, _) in segments[sid]["pos"].items():
            if a <= num <= b:
                positions.setdefault(num, (bb, "lu"))
for sid, a, b in raccords:
    spans.append((KO + a / FPS, KO + b / FPS))
    etiquette = ",".join(ABREGE[v] for v in VAR_PAR_SEG[sid] if v != "B partiel (v2.4)")   # ex. « R,B »
    for num, (bb, _) in segments[sid]["pos"].items():
        if a <= num <= b and positions.get(num, (None, ""))[1] != "lu":
            positions[num] = (bb, "raccord:" + etiquette)

# ── v3.1 : TOUCHES DE BALLE du joueur, d'après les événements du suivi ─────────────────────────────
_tids_lus = collections.defaultdict(set)          # image → pistes du joueur sur une portion où son numéro est lu
for sid in retenus:
    _tid = int(sid.split("@")[0])
    for a, b in verdicts[sid][0]:
        for num in segments[sid]["pos"]:
            if a <= num <= b:
                _tids_lus[num].add(_tid)
_cles_ev = [k for k in ("tracker_id", "track_id", "player_track_id", "tid") if events and isinstance(events[0], dict)
            and any(k in e for e in events[:2000])]
print(f"\n   TOUCHES : champs de piste trouvés dans les événements : {_cles_ev} | exemple : "
      f"{ {k: events[0][k] for k in list(events[0])[:10]} if events and isinstance(events[0], dict) else type(events[0]) }")
TOUCHES = []
_pas = 5                                           # le suivi garde une image sur 5
_ev_ok = [e for e in (events if isinstance(events, list) else []) if isinstance(e, dict)
          and str(e.get("type", "")) in TYPES_TOUCHES and e.get("frame") is not None]
_cles_pos = np.array(sorted(positions)) if positions else np.array([], dtype=int)


def _bb_proche(n):
    if not len(_cles_pos):
        return None
    i = int(np.searchsorted(_cles_pos, n))
    best = None
    for j in (i - 1, i):
        if 0 <= j < len(_cles_pos) and abs(int(_cles_pos[j]) - n) <= _pas:
            best = positions[int(_cles_pos[j])][0]
    return best


def _touches(decal, echelle):
    out = []
    for e in _ev_ok:
        n = int(e["frame"]) + decal
        if _cles_ev:
            pistes = _tids_lus.get(n) or set()
            if pistes and any(e.get(k) is not None and int(e.get(k)) in pistes for k in _cles_ev):
                out.append((KO + n / FPS, str(e.get("type"))))
            continue
        if e.get("x") is None or e.get("y") is None:
            continue
        bb = _bb_proche(n)
        if bb is None:
            continue
        x, y = float(e["x"]) * echelle, float(e["y"]) * echelle
        x1, y1, x2, y2 = bb; mx, my = 0.3 * (x2 - x1), 0.3 * (y2 - y1)
        if x1 - mx <= x <= x2 + mx and y1 - my <= y <= y2 + my:
            out.append((KO + n / FPS, str(e.get("type"))))
    return out


# 08/11 : la numérotation des images et l'échelle des coordonnées des événements ne sont pas documentées → on essaie
# les combinaisons plausibles et on garde celle qui rattache le plus d'événements (affichée pour contrôle).
_essais = {}
for _d in (0, -int(round(KO * FPS))):
    for _e in (1.0, 1920 / 853):
        _essais[(_d, _e)] = _touches(_d, _e)
(_dbest, _ebest), TOUCHES = max(_essais.items(), key=lambda kv: len(kv[1]))
if _ev_ok:
    print(f"   TOUCHES : {len(_ev_ok)} événement(s) de jeu | images du suivi {int(_cles_pos.min()) if len(_cles_pos) else '-'}"
          f"-{int(_cles_pos.max()) if len(_cles_pos) else '-'} / événements {min(int(e['frame']) for e in _ev_ok)}-{max(int(e['frame']) for e in _ev_ok)}"
          f" | x max {max(float(e.get('x') or 0) for e in _ev_ok):.0f} | rattachés : "
          + ", ".join(f"décalage {d} éch. {e:.2f} → {len(v)}" for (d, e), v in _essais.items()))
TOUCHES.sort()
_fus = []
for t, ty in TOUCHES:                              # une touche = événements à moins de 2 s regroupés
    if _fus and t - _fus[-1][1] <= 2.0:
        _fus[-1][1] = t; _fus[-1][2].add(ty)
    else:
        _fus.append([t, t, {ty}])
print(f"   TOUCHES du {NUMERO} {COULEUR} : {len(TOUCHES)} événement(s) → {len(_fus)} touche(s) regroupée(s)")
for a, b, tys in _fus[:80]:
    print(f"     {fmt(a)}{'-' + fmt(b) if b - a >= 1 else ''} : {', '.join(sorted(tys))}")
with open(f"{SORTIE}/touches_v31_{MATCH}_{NUMERO}_{COULEUR.replace(' ', '_')}.csv", "w", encoding="utf-8") as _f:
    _f.write("debut;fin;types\n" + "".join(f"{fmt(a)};{fmt(b)};{'+'.join(sorted(t))}\n" for a, b, t in _fus))
# ── joueur seul : BUTS où le joueur est IMPLIQUÉ ; les autres (ex. un but du 9 rouge sans lui) sont ignorés ──
# Impliqué = une de SES touches dans les 15 s avant le but (but, passe, centre, duel…) OU reconnu dans les 8 s avant
# (dans l'action : buteur de la tête, défenseur sur le tireur…). Simplement visible 20 s avant ne suffit plus.
_buts_joueur = []
if BOUT_EN_BOUT and BOUT_EN_BOUT.get("buts"):
    _t_pos = np.array(sorted(KO + n / FPS for n in positions)) if positions else np.array([])
    print(f"\n   BUTS du match ({len(BOUT_EN_BOUT['buts'])}) — TOUS dans le résumé du joueur (implication indiquée pour information) :")
    for b in BOUT_EN_BOUT["buts"]:
        g = float(b["time_abs"])
        vu = float(((_t_pos >= g - 8) & (_t_pos <= g + 2)).sum()) / 6.0 if len(_t_pos) else 0.0   # ≈ secondes (6 images/s)
        touche = [t for t, _ in TOUCHES if g - 15 <= t <= g + 1]
        garde = bool(touche) or vu >= 1.0
        raison = (f"{len(touche)} touche(s) dans les 15 s avant" if touche else f"reconnu ≈ {vu:.0f} s dans les 8 s avant") if garde \
            else "ignoré (pas impliqué, ou pas reconnu)"
        # 08/11 (décision de l'utilisateur) : TOUS les buts du match entrent dans le résumé du joueur, pour ne jamais
        # rater un but où il est buteur ou impliqué ; l'indication « impliqué » n'est plus qu'une information.
        print(f"     but {fmt(g)} ({b.get('statut')}) → dans le résumé | {'impliqué : ' + raison if garde else 'implication non détectée'}")
        _buts_joueur.append(g)
# ── joueur seul : CE QUE L'ON SAIT DE SON MATCH (données disponibles) ──
_types = collections.Counter(ty for _, ty in TOUCHES)
_presence = len(positions) / 6.0 if positions else 0.0
print(f"\n   DONNÉES DU {NUMERO} {COULEUR} : présence reconnue ≈ {_presence / 60:.1f} min | {len(_fus)} action(s) avec ballon "
      f"({len(TOUCHES)} événements : " + ", ".join(f"{k} {v}" for k, v in _types.most_common()) + f") | {len(_buts_joueur)} but(s) du match dans le résumé")
print("   (fautes : NON détectées par le suivi — à ajouter plus tard si utile)")
if MONTAGE == "touches" and (_fus or _buts_joueur):
    _FEN = sorted([(a - TOUCHE_AVANT, b + TOUCHE_APRES) for a, b, _ in _fus] + [(g - 25.0, g + 5.0) for g in _buts_joueur])   # 25 s avant : l heure du but du pipeline peut être en retard (jusqu à ~20 s)
    print(f"   montage : {len(_fus)} touche(s) + {len(_buts_joueur)} but(s) avec le joueur")
elif MONTAGE == "touches":
    print("   ⚠️ aucune touche ni aucun but rattaché au joueur : montage de présence (v2.8)")
    _FEN = []                                      # sinon le montage reprendrait les mi-temps entières

# ── 5. (suite) CONSENSUS DANS LE TEMPS (option, désactivée par défaut) ───────────────────────────
CONS_GROUPES, CONS_AJOUT = [], 0
if CONSENSUS_TEMPORAL:
    renf = {}
    for _c in sorted(glob.glob(f"/kaggle/input/**/lectures_renforcees_{MATCH}.json", recursive=True)) + \
              [f"{SORTIE}/lectures_renforcees_{MATCH}.json"]:
        if os.path.exists(_c):
            try:
                renf.update(json.load(open(_c, encoding="utf-8")))
            except Exception as _e:
                print(f"   ⚠️ {_c} illisible ({_e})")
    autre_eq = 1 - EQ
    obs = []
    for src_l, base in (("normale", lectures), ("renforcée", renf)):
        for sid, e in base.items():
            if sid not in segments or e.get("numeros") is None or e.get("non_lu"):
                continue
            for n, num, p in zip(e.get("images", []), e["numeros"], e.get("parts", [])):
                if n in segments[sid]["pos"]:
                    obs.append((int(n), sid, num, equipe_mesuree(p), tuple(segments[sid]["pos"][n][0]), src_l))
    obs = sorted(set(obs), key=lambda o: (o[0], o[1], o[5], -1 if o[2] is None else o[2], -1 if o[3] is None else o[3]))
    dix = [o for o in obs if o[2] == NUMERO and o[3] != autre_eq]

    def _dist_h(b1, b2):
        h = max(1.0, ((b1[3] - b1[1]) + (b2[3] - b2[1])) / 2)
        return float(np.hypot((b1[0] + b1[2]) / 2 - (b2[0] + b2[2]) / 2, b1[3] - b2[3]) / h)

    def _lies(o1, o2):
        dt = abs(o1[0] - o2[0]) / FPS
        return dt <= CONS_S and _dist_h(o1[4], o2[4]) <= CONS_BASE_H + CONS_H_PAR_S * dt

    parent = list(range(len(dix)))
    def _rac(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]; i = parent[i]
        return i
    for i in range(len(dix)):
        for j in range(i + 1, len(dix)):
            if (dix[j][0] - dix[i][0]) / FPS > CONS_S:
                break
            if _lies(dix[i], dix[j]):
                parent[_rac(j)] = _rac(i)
    groupes = collections.defaultdict(list)
    for i in range(len(dix)):
        groupes[_rac(i)].append(dix[i])
    coupures = collections.defaultdict(list)
    for o in obs:
        if o[3] == autre_eq or (o[2] is not None and o[2] != NUMERO):
            coupures[o[1]].append(o[0])
    deja = set(positions)
    ajout = {}
    for k, g in enumerate(sorted(groupes.values(), key=lambda g: g[0][0]), 1):
        # lectures simultanées à deux endroits : exclure la moins soutenue (les deux si égalité)
        exclues = set()
        for i in range(len(g)):
            for j in range(i + 1, len(g)):
                if abs(g[i][0] - g[j][0]) <= 0.2 * FPS and _dist_h(g[i][4], g[j][4]) > 0.5:
                    si = sum(1 for o in g if o is not g[i] and _lies(o, g[i]))
                    sj = sum(1 for o in g if o is not g[j] and _lies(o, g[j]))
                    if si <= sj: exclues.add(i)
                    if sj <= si: exclues.add(j)
        garde = [o for i, o in enumerate(g) if i not in exclues]
        n_rouges = sum(1 for o in garde if o[3] == EQ)
        valide = len(garde) >= CONS_MIN_LECTURES and n_rouges >= CONS_MIN_ROUGES
        nouv = set()
        if valide:
            for n, sid, *_ in garde:
                lo, hi = n - GARDE_S * FPS, n + GARDE_S * FPS
                for c in coupures.get(sid, []):
                    if c < n: lo = max(lo, c + 1)
                    elif c > n: hi = min(hi, c - 1)
                for m in segments[sid]["pos"]:
                    if lo <= m <= hi and m not in deja and m not in ajout:
                        ajout[m] = segments[sid]["pos"][m][0]; nouv.add(m)
        statut = ("ajouté" if nouv else "déjà couvert") if valide else "faible (non ajouté)"
        if len(g) >= 2 or exclues:
            CONS_GROUPES.append({"groupe": k, "debut": fmt(KO + g[0][0] / FPS), "fin": fmt(KO + g[-1][0] / FPS),
                                 "lectures": len(garde), "rouges": n_rouges, "fragments": len({o[1] for o in garde}),
                                 "exclues_conflit": len(exclues), "statut": statut,
                                 "secondes_ajoutees": len({int(KO + m / FPS) for m in nouv}),
                                 "detail": " ".join(f"{fmt(KO + o[0] / FPS)}:{o[1]}:{'R' if o[3] == EQ else '?'}:{o[5][:4]}"
                                                    for o in garde),
                                 "_obs": garde})
    for m, bb in ajout.items():
        positions[m] = (bb, "consensus")
    blocs, cur = [], []
    for m in sorted(ajout):
        if cur and m - cur[-1] > 0.5 * FPS:
            blocs.append(cur); cur = []
        cur.append(m)
    if cur:
        blocs.append(cur)
    for bl in blocs:
        spans.append((KO + bl[0] / FPS, KO + bl[-1] / FPS))
    CONS_AJOUT = len({int(KO + m / FPS) for m in ajout})
    st = collections.Counter(gr["statut"] for gr in CONS_GROUPES)
    print(f"\n   CONSENSUS DANS LE TEMPS : {len(obs)} vignettes lues ({sum(1 for o in obs if o[5] == 'renforcée')} renforcées) | "
          f"groupes {dict(st)} | lectures exclues pour conflit : {sum(gr['exclues_conflit'] for gr in CONS_GROUPES)} | "
          f"présence ajoutée : {CONS_AJOUT} s")
    with open(f"{SORTIE}/consensus_v28_{MATCH}_{NUMERO}_{COULEUR}.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=[c for c in CONS_GROUPES[0] if c != "_obs"] if CONS_GROUPES else ["groupe"],
                           delimiter=";", extrasaction="ignore")
        w.writeheader(); w.writerows(CONS_GROUPES)
    lignes_c = []
    for gr in CONS_GROUPES:
        if gr["statut"] != "ajouté":
            continue
        cases = []
        for o in gr["_obs"][:7]:
            ok, img = image_video(o[0])
            if ok and decoupe(img, o[4]) is not None:
                cases.append(case(decoupe(img, o[4]), f"{fmt(KO + o[0] / FPS)} {'R' if o[3] == EQ else '?'}", o[3] == EQ))
        if cases:
            lignes_c.append(ligne_planche(f"groupe {gr['groupe']} | {gr['debut']}-{gr['fin']} | {gr['lectures']} lectures, "
                                          f"{gr['rouges']} rouges, {gr['exclues_conflit']} exclue(s) | +{gr['secondes_ajoutees']} s",
                                          (0, 140, 255), cases))
    if lignes_c:
        ecrire_planches(lignes_c, f"{SORTIE}/verification_consensus_v30")
        print(f"   planche : verification_consensus_v30_p*.jpg ({len(lignes_c)} groupe(s) ajouté(s), à vérifier)")
cles_pos = np.array(sorted(positions))
sequences = []
if _FEN:                                  # v2.9 : le résumé complet, repère là où le joueur est reconnu
    spans = list(_FEN)
for a, b in sorted(spans):
    if sequences and a - sequences[-1][1] <= FUSION_S:
        sequences[-1][1] = max(sequences[-1][1], b)
    else:
        sequences.append([a, b])


def cadre_a(t_video):
    num = (t_video - KO) * FPS
    i = int(np.argmin(np.abs(cles_pos - num)))
    if abs(cles_pos[i] - num) > 0.3 * FPS:
        return None
    return positions[int(cles_pos[i])]


def extrait_avec_repere(a, b, sortie):
    cap_e = cv2.VideoCapture(VIDEO)
    fps_v = cap_e.get(cv2.CAP_PROP_FPS) or FPS
    cap_e.set(cv2.CAP_PROP_POS_MSEC, a * 1000)
    ok, img = cap_e.read()
    if not ok:
        cap_e.release(); return False, 0.0, 0.0
    H, W = img.shape[:2]
    tmp = sortie + ".brut.mp4"
    vw = cv2.VideoWriter(tmp, cv2.VideoWriter_fourcc(*"mp4v"), fps_v, (W, H))
    t, dernier, t_dernier, n_lu, n_rac, n_cons = a, None, None, 0, 0, 0
    while ok and t <= b:
        x = cadre_a(t)
        if x is not None:
            dernier, t_dernier = x, t                   # seule une VRAIE position remet le compteur à zéro
        elif dernier is not None and t - t_dernier <= TENUE_REPERE_S:
            x = dernier                                 # petit trou : repère tenu au plus TENUE_REPERE_S
        if x is not None:
            bb, source = x
            est_rac = source.startswith("raccord")
            est_cons = source == "consensus"
            rac = est_rac and MARQUER_RACCORDS
            n_rac += est_rac; n_cons += est_cons; n_lu += not (est_rac or est_cons)
            coul = (255, 160, 0) if rac else (0, 140, 255) if est_cons else (0, 230, 255)   # bleu / orange / jaune
            x1, y1, x2, y2 = [int(v) for v in bb]
            cx, haut = (x1 + x2) // 2, max(30, y1 - 12)
            cv2.drawContours(img, [np.array([[cx - 14, haut - 24], [cx + 14, haut - 24], [cx, haut]])], 0, coul, -1)
            txt = f"#{NUMERO}" + (f"? {source.split(':', 1)[1]}".rstrip() if rac else "*" if est_cons else "")
            cv2.putText(img, txt, (cx - 22, haut - 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, coul, 2)
            cv2.ellipse(img, (cx, y2), (max(14, (x2 - x1) // 2 + 6), 8), 0, 0, 360, coul, 2)
        vw.write(img)
        ok, img = cap_e.read()
        t += 1.0 / fps_v
    vw.release(); cap_e.release()
    r = subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", tmp, "-ss", f"{a:.2f}", "-t", f"{b - a:.2f}", "-i", VIDEO,
                        "-map", "0:v", "-map", "1:a?", "-c:v", "libx264", "-preset", "veryfast", "-crf", "21",
                        "-c:a", "aac", "-shortest", "-movflags", "+faststart", sortie], capture_output=True)
    os.remove(tmp)
    return r.returncode == 0 and os.path.exists(sortie), n_lu / fps_v, n_rac / fps_v, n_cons / fps_v


# ── 5 bis. CONTRÔLE DES BUTS (vérité terrain) : le joueur est-il repéré juste avant chaque but ? ─────────
CONTROLE_BUTS = []
_vt = sorted(glob.glob("/kaggle/input/**/verite_terrain.json", recursive=True)) + \
      sorted(glob.glob(f"{SORTIE}/verite_terrain.json"))
_buts, _buteurs = [], {}
for _c in _vt:
    try:
        _m = json.load(open(_c, encoding="utf-8"))["matchs"][MATCH]
        _buts, _buteurs = _m.get("buts", []), _m.get("buteurs", {})
        break
    except Exception:
        pass
if not _buts:
    print("\n   contrôle des buts : verite_terrain.json absent ou sans ce match (ajoute-le en entrée) — ignoré")
else:
    _sec = lambda t: int(t.split(":")[0]) * 60 + float(t.split(":")[1])
    print(f"\n   CONTRÔLE DES BUTS ({len(_buts)}) : présence du {NUMERO} {COULEUR} dans les {CONTROLE_BUT_AVANT_S} s avant chaque but")
    for _b in _buts:
        tb = _sec(_b)
        n0, n1 = (tb - CONTROLE_BUT_AVANT_S - KO) * FPS, (tb + 1 - KO) * FPS
        dans = [n for n in positions if n0 <= n <= n1]
        s_lu = sum(1 for n in dans if positions[n][1] == "lu") * 5 / FPS
        s_rac = sum(1 for n in dans if positions[n][1].startswith("raccord")) * 5 / FPS
        s_cons = sum(1 for n in dans if positions[n][1] == "consensus") * 5 / FPS
        dernier = fmt(KO + max(dans) / FPS) if dans else "-"
        # segments de l'équipe dans la fenêtre, avec leur état
        etat = collections.Counter()
        en_attente = []
        for _, sid, _ in candidats:
            a_, b_ = min(segments[sid]["pos"]), max(segments[sid]["pos"])
            if b_ < n0 or a_ > n1:
                continue
            e = lectures.get(sid) or {}
            st = "retenu" if sid in retenus else e.get("non_lu") or ("lu" if e.get("numeros") is not None else "non examiné")
            etat[st] += 1
            if st in ("plafond", "non examiné"):
                en_attente.append(f"{fmt(KO + a_ / FPS)}-{fmt(KO + b_ / FPS)}")
        info = _buteurs.get(_b, {})
        qui = f"{info.get('equipe', '?')} {info.get('numero', '?')}" if info else "?"
        CONTROLE_BUTS.append([_b, qui, info.get("remarque", ""), round(s_lu, 1), round(s_rac, 1), dernier,
                              dict(etat), " ".join(en_attente)])
        print(f"     but {_b} (buteur : {qui}) : {NUMERO} repéré {s_lu:.0f} s (lu) + {s_rac:.0f} s (raccord)"
              + (f" + {s_cons:.0f} s (consensus)" if CONSENSUS_TEMPORAL else "") + ", "
              f"dernière position {dernier} | segments rouges de la fenêtre : {dict(etat)}"
              + (f" | encore à lire : {', '.join(en_attente)}" if en_attente else ""))
    with open(f"{SORTIE}/controle_buts_v30_{MATCH}_{NUMERO}_{COULEUR}.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f, delimiter=";")
        w.writerow(["but", "buteur", "remarque", "repere_lu_s", "repere_raccord_s", "derniere_position",
                    "segments_rouges_fenetre", "segments_a_lire"])
        w.writerows(CONTROLE_BUTS)

dossier_clips = f"{SORTIE}/clips_v28_{NUMERO}_{COULEUR}"
os.makedirs(dossier_clips, exist_ok=True)
for ancien in glob.glob(f"{dossier_clips}/seq_*.mp4"):
    os.remove(ancien)
clips, lignes = [], []
for i, (a0, b0) in enumerate(sequences, 1):
    a, b = max(0.0, a0 - MARGE_AVANT), b0 + MARGE_APRES
    clip = f"{dossier_clips}/seq_{i:02d}.mp4"
    ok_c, s_lu, s_rac, s_cons = extrait_avec_repere(a, b, clip)
    if ok_c:
        clips.append(clip)
    lignes.append([i, fmt(a), fmt(b), f"{max(0, a0 - KO) / 60:.0f}'", round(b - a, 1), round(s_lu, 1), round(s_rac, 1),
                   round(s_cons, 1)])
    print(f"     séquence {i:2d} : vidéo {fmt(a)} → {fmt(b)} ({b - a:.0f} s, repère lu {s_lu:.0f} s, raccord {s_rac:.0f} s"
          + (f", consensus {s_cons:.0f} s" if CONSENSUS_TEMPORAL else "") + ")")
for c in _caps:
    c.release()
suffixe = "_essai" if MODE == "essai" else ""
montage = f"{SORTIE}/montage_v30_{MATCH}_{NUMERO}_{COULEUR}{suffixe}.mp4"
if clips:
    with open(f"{dossier_clips}/liste.txt", "w") as f:
        f.writelines(f"file '{c}'\n" for c in clips)
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-f", "concat", "-safe", "0", "-i", f"{dossier_clips}/liste.txt", "-c", "copy", montage], capture_output=True)
with open(f"{SORTIE}/montage_v30_{MATCH}_{NUMERO}_{COULEUR}{suffixe}.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f, delimiter=";")
    w.writerow(["sequence", "debut_video", "fin_video", "minute_de_jeu", "duree_s", "repere_lu_s", "repere_raccord_s",
                "repere_consensus_s"])
    w.writerows(lignes)

# ── BILAN ─────────────────────────────────────────────────────────────────────────────────────
duree = sum(l[4] for l in lignes)
minutes = sorted({l[3] for l in lignes}, key=lambda m: int(m[:-1]))
print(f"\n{'═' * 90}\nBILAN v2.8 du {NUMERO} {COULEUR} ({MATCH}, mode {MODE})")
print(f"   lectures fusionnées : {len(SOURCES_L)} fichier(s) | conflits journalisés : {len(CONFLITS)}")
print(f"   segments candidats : {N_CANDIDATS_TOTAL} | examinés : {len(candidats)} | lectures réutilisées : {N_REUTILISEES_ICI} | "
      f"questions évitées déjà connues : {N_EVITEES_CONNUES}")
print(f"   NOUVELLES questions Gemini posées : {compteur['questions']} | évitées par la couleur : {compteur['evitees']} | "
      f"en attente (plafond) : {compteur['plafond']} | erreurs : {len(erreurs)}")
print(f"   segments retenus par lecture : {len(retenus)} | raccords ({RACCORD_MONTAGE}, montage) : {len(raccords)} | "
      + " | ".join(f"{c[0]} : {c[1]}" for c in COMPARAISON))
print(f"   montage : {len(clips)} séquence(s), {duree / 60:.1f} min | repère lu {sum(l[5] for l in lignes):.0f} s, "
      f"raccord {sum(l[6] for l in lignes):.0f} s" + (f", consensus {sum(l[7] for l in lignes):.0f} s" if CONSENSUS_TEMPORAL else "")
      + f" | minutes de jeu : {', '.join(minutes)}")
print(f"   consensus dans le temps : {'ACTIVÉ' if CONSENSUS_TEMPORAL else 'désactivé (comportement v2.7)'}"
      + (f" | groupes ajoutés : {sum(1 for g in CONS_GROUPES if g['statut'] == 'ajouté')}, faibles écartés : "
         f"{sum(1 for g in CONS_GROUPES if g['statut'].startswith('faible'))}, présence ajoutée {CONS_AJOUT} s"
         if CONSENSUS_TEMPORAL else ""))
print(f"   erreurs de raccordement : NON mesurables automatiquement → vérifier les repères BLEUS du montage et "
      f"verification_raccords_v28_p*.jpg\n   {montage if os.path.exists(montage) else '(aucun montage)'}\n"
      f"   Durée du notebook : {(time.time() - T0) / 60:.1f} min")
cout_d = dollars(COUT)
print(f"   réflexion de la lecture des numéros : {REFLEXION_LECTURE or 'normale'}")
print(f"   COÛT GEMINI de ce run (mesuré) : {COUT['questions']} question(s) | jetons entrée {COUT['entree']:,} | sortie "
      f"{COUT['sortie']:,} | réflexion {COUT['reflexion']:,} | ≈ {cout_d:.3f} $ (≈ {cout_d * TAUX_EUR:.3f} €)"
      + (f" | ⚠️ {COUT['sans_mesure']} réponse(s) sans mesure" if COUT["sans_mesure"] else ""))
for usage, c in COUT_PAR_USAGE.items():
    print(f"      {usage} : {c['questions']} question(s), ≈ {dollars(c):.3f} $"
          + (f" ({dollars(c) / c['questions'] * 1000:.2f} $ pour 1 000 questions)" if c["questions"] else ""))
print("═" * 90)
# journal des coûts : une ligne par run, conservé d'un run à l'autre s'il est remis en entrée
journal = f"{SORTIE}/cout_gemini_{MATCH}.csv"
anciens = sorted(glob.glob(f"/kaggle/input/**/cout_gemini_{MATCH}.csv", recursive=True))
if not os.path.exists(journal) and anciens:
    import shutil
    shutil.copy(anciens[-1], journal)
nouveau = not os.path.exists(journal)
with open(journal, "a", newline="", encoding="utf-8") as f:
    w = csv.writer(f, delimiter=";")
    if nouveau:
        w.writerow(["date", "version", "mode", "questions", "jetons_entree", "jetons_sortie", "jetons_reflexion",
                    "cout_dollars", "cout_euros_approx"])
    w.writerow([time.strftime("%Y-%m-%d %H:%M"), VERSION, MODE, COUT["questions"], COUT["entree"], COUT["sortie"],
                COUT["reflexion"], round(cout_d, 4), round(cout_d * TAUX_EUR, 4)])
print(f"   journal des coûts : {journal} (une ligne par run)")


# ── BOUT EN BOUT : bilan global ──
if BOUT_EN_BOUT:
    print(f"\n{'═' * 90}\nBILAN DE BOUT EN BOUT — {MATCH}, joueur n°{NUMERO} {COULEUR}")
    print(f"   durée : analyse du match {BOUT_EN_BOUT.get('duree_analyse_min')} min + montage du joueur "
          f"{(_tbe.time() - T_DEBUT_MONTAGE) / 60:.0f} min")
    print("   coût Gemini : analyse = « [GEMINI CACHE] … payée(s) » + « coût tenues + attribution » (cellule 1) ;"
          " joueur = « COÛT GEMINI de ce run » + « CONFIRMATION GEMINI » (ci-dessus)")
    print(f"   livrables : rapport PDF (cellule 1), montage {SORTIE}/montage_v30_{MATCH}_{NUMERO}_{COULEUR}.mp4\n{'═' * 90}")


# ════════════════════════════════════════════════════════════════════════════════════════════
# BILAN DE PRODUCTION — un match, de la vidéo brute aux livrables
# ════════════════════════════════════════════════════════════════════════════════════════════
_duree_montage = (time.time() - T_DEBUT_MONTAGE) / 60
_cout_an = sum(v[1] for v in COUT_ANALYSE.values())
_cout_j = cout_d if "cout_d" in dir() else 0.0
print(f"\n{'█' * 90}\nBILAN DE PRODUCTION — {MATCH} (vidéo {os.path.getsize(VIDEO) / 1e9:.1f} Go)\n{'█' * 90}")
print(f"   TEMPS   analyse du match {DUREE_ANALYSE_MIN:6.1f} min | montage du joueur n°{NUMERO} {_duree_montage:6.1f} min | "
      f"TOTAL {DUREE_ANALYSE_MIN + _duree_montage:6.1f} min")
print(f"   GEMINI  analyse {_cout_an:.2f} $ (dont équipes {sum(v[1] for f, v in COUT_ANALYSE.items() if f == '__call__'):.2f} $) | "
      f"joueur {_cout_j:.2f} $ | TOTAL {_cout_an + _cout_j:.2f} $ (≈ {(_cout_an + _cout_j) * 0.86:.2f} € au taux indicatif 0,86, à vérifier)")
print(f"   GPU     ≈ {(DUREE_ANALYSE_MIN + _duree_montage) / 60:.2f} h de T4 ×2 (coût à évaluer selon l'hébergeur de production)")
print(f"   ÉQUIPES {_att.get('tenues') if _att else '—'} | score confirmé {_att.get('score_confirme') if _att else '—'} | "
      f"estimé {_att.get('score_estime') if _att else '—'}")
print(f"   LIVRABLES rapport /kaggle/working/rapport_{MATCH}.pdf | montage {SORTIE}/montage_v30_{MATCH}_{NUMERO}_{COULEUR}.mp4 | "
      f"cache de suivi /kaggle/working/cache_{MATCH}_reference.zip")
print(f"   (comparer : tarifs Pro 2,69 € / Unique 5 €)\n{'█' * 90}")

  Clés lues dans /kaggle/input/datasets/michaelziant/mziparameter/param.env : SECRET_KEY, CLAUDE_API_KEY, GEMINI_API_KEY, STRIPE_SECRET_KEY, STRIPE_PUBLIC_KEY, R2_SECRET_ACCESS_KEY, RUNPOD_API_KEY

ÉTAPE 1/3 — Clone


Cloning into '/kaggle/working/ProjectAIMika'...


Clone OK

ÉTAPE 2/3 — Dépendances
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.7/92.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.6/360.6 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.1/85.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.9/102.9 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.
Dépendances OK

ÉTAPE 3/3 — Environnement
Environnement OK

CONTRÔLE PRÉ-VOL
  ✅ KO start_s explicite (replay OK)
  ✅ _ko_goals compare aux buts seulement
  ✅ pipeline : statut des buts + KO2 + temps forts
  ✅ pipeline : contrôle final des doublons
  ✅ pipeline : mi-temps en parallèle (bornes)
  ✅ pipeline : règle de la pause SÛRE (28/10)
  ✅ pipeline : voie « ballon » désactivée par défaut
  ✅ pipeline : second regard activé par défaut
  ✅ pipeline : buts avec statut + identification
  ✅ pipeline : doublons de temps forts quelle que soit l'issue (31/10)
  ✅ export/pdf : frise des Top Highlights sur plusieurs lignes (31/10)
  ✅ gemini_validator : réflexion réduite (KO, classification)
  ✅ gemini_validator : modèle u

/usr/local/lib/python3.12/dist-packages/torchreid/reid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(
E0000 00:00:1794429301.120252    1339 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1794429301.170689    1339 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1794429301.598606    1339 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1794429301.598637    1339 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1794429301.598640    1339 computation_placer.cc:177] computation placer already regis

  Detector SINGLE YOLO26m : yolo26m.pt | joueurs+ballon | conf_ballon=0.15
  Detector pret : yolo26m.pt | sport=football | camera_type=high_side | proximite_poids=0.5 | seuil_gap_protection=None | intervalle_recherche_globale=None | activer_multi_hypotheses=False | seuil_streak_gele=None
  Tracker : DeepSort (mobilenet FP16 GPU)
  Tracker : mode HISTORIQUE — toutes les pistes confirmées transmises
  [OCR] DÉSACTIVÉ — test performance tracking pur
  BallTracker : OK
  [BALL TRACKER] fps effectif fixé à 6.0 (natif=30.0, skip_every=5)
  [OCR] désactivé (DISABLE_OCR=1)
  [START_TIME_S] Positionnement à t=713.0s (frame 21368/217878) — frames précédentes non lues, non traitées
  [END_TIME_S] Lecture arrêtée à t=4023.0s (frame 120569/217878) — frames suivantes non lues, non traitées
Video : /kaggle/input/datasets/michaelziant/mineroisster-complet/minerois-ster-2025-11-09.mp4
  217878 frames | 30.0 fps | 7269.9s
  Resolution : 1920x1080 → traitement 960x540
  Sport : football
  Frame skip  : 5

/usr/local/lib/python3.12/dist-packages/torchreid/reid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(
E0000 00:00:1794430125.413602    3584 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1794430125.420158    3584 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1794430125.439759    3584 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1794430125.439785    3584 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1794430125.439788    3584 computation_placer.cc:177] computation placer already regis

  [FIN1MT_GEMINI] Q1 à t=3431s : NON — [signal=aucun] non fourni
  [FINMATCH_GEMINI] histoire : CONTREDITE — Scène identique, aucun mouvement de sortie ni salutations, simple arrêt de jeu statique.
  [FINMATCH_GEMINI] candidat rejeté par le narratif, prochain signal isolé tenté si trouvé
  [FINMATCH_GEMINI] Q1 à t=6791s : NON — [signal=aucun] non fourni
  [FIN1MT_GEMINI] Q1 à t=3491s : SORTIE — [signal=A] non fourni
  [FIN1MT_GEMINI] Q1 à t=3551s : SORTIE — [signal=A] non fourni  [FINMATCH_GEMINI] Q1 à t=6851s : NON — [signal=aucun] non fourni

  [FINMATCH_GEMINI] Q1 à t=6911s : SIGNAL_SORTIE — [signal=A] non fourni
  [FINMATCH_GEMINI] vérification narrative de secours (t=6911s, t+5s=6916s)...
  [FIN1MT_GEMINI] Q1 à t=3611s : SORTIE — [signal=B] non fourni
  [FIN1MT_GEMINI] pause établie (3 confirmations Q1 consécutives), 1er point=3491s — recherche fine dans [3431s, 3491s]...
  [FINMATCH_GEMINI] histoire : CONTREDITE — Joueur plaçant le ballon pour coup franc, arbitre actif, simple ar

100%|██████████| 91.0M/91.0M [00:00<00:00, 211MB/s]


   lecteur de numéros : PARSeq affiné (/kaggle/input/datasets/michaelziant/codeprojectai/parseq_scoutia_maillots_v2.pt) sur cuda, seuil 0.8
   lectures : 0 segment(s) déjà connu(s) | 3048 à examiner (couleur d'abord ; question seulement si ≥ 2 vignettes « bleu clair », au plus 1000000000)
   LECTURE RAPIDE : 3048 morceau(x), 32019 images lues en 818 s (vidéo 597 s en 2 processus, PARSeq 221 s) — les deux équipes, réutilisable pour tout joueur du match
   lectures sauvegardées : /kaggle/working/lectures_v30_minerois_ster_v2.json (à garder : sortie du notebook, puis dataset en entrée)
   CONFIRMATION GEMINI : 17 segment(s) lus une seule fois | 17 relu(s) | 8 désormais retenu(s) | ≈ 0.074 $
   RETENUS (numéro lu) pour le 16 bleu clair : 71 segment(s) | écartés : {'aucun numéro lisible': 600, 'autre numéro': 2368, '16 lu, mais aucune vignette de la bonne couleur': 2, '16 lu moins de 2 fois sur des vignettes consécutives de la bonne couleur sans autre numéro': 7}
   planche des lectures : 1